# How does raw data become training data, features and a live stream without leaks or lost records?

Question: where can a data pipeline lose a record, copy a record, or let the future leak into a feature? And how do you measure each fault, instead of trusting that the pipeline is right?

Setup: Kaggle CPU run (the GPUs stay idle), Python 3.13, internet on, about 25 minutes. The notebook needs no API key. Parts A to G, I and J run in the kernel with numpy, pandas, scikit-learn and sqlite3. Part E also runs Feast, part H also runs a real Kafka broker and Spark. Those three stacks live in their own `uv venv` and run as subprocesses, so they cannot upgrade a package that the kernel already imported. Each stack has a Python fallback. If a stack does not start, the part prints the real error and the fallback numbers. All randomness uses fixed seeds. Every rate has a Wilson 95% interval. Two systems on the same items get an exact McNemar test.

Exam objectives: data.1 (feature engineering, feature stores, exploratory data analysis), data.3 (data ingestion, ETL, distributed processing, Spark, Kafka), data.4 (data cleaning, data labeling, text preprocessing), data.5 (data retention, data sovereignty, bias, misuse risks), data.6 (structured, semi-structured, unstructured data), data.10 (training, inference and retrieval data flows, batch versus streaming).

Two parts rebuild questions that were answered wrong before:

- **Part D1**: a feature built from the target mean on all rows, then cross-validation. The score is far above the next month's score.
- **Part D2**: eight columns with missing values, where the pattern of missing columns predicts the target. Which single column captures the pattern?

`data-flywheel-cudf.ipynb` already covers PII redaction, freshness checks and quality gates. This notebook does not repeat them.

Terms used below:

- **Structured data**: rows and columns with a fixed schema, such as a SQL table. **Semi-structured data**: records with a tree shape and no fixed schema, such as nested JSON logs. **Unstructured data**: free text, such as an email.
- **Ingestion**: reading data from a source into your system. **ETL** (extract, transform, load): clean and shape the data first, then load it. **ELT**: load the raw data first, then transform it inside the target.
- **Dead-letter queue (DLQ)**: a store for records that failed validation. Nothing is dropped in silence.
- **Idempotency key**: an ID that makes a repeated write safe. The second write of the same ID changes nothing.
- **Exploratory data analysis (EDA)**: look at the data with code before you model it.
- **Target leakage**: a feature holds information that is not available when the model predicts.
- **Target mean (target encoding)**: replace a category by the mean of the target in that category.
- **Feature store**: a system that keeps one definition of each feature, an offline copy for training and an online copy for serving. **Point-in-time join**: for each training row, take the feature value that was known at that row's time. **Training-serving skew**: the training pipeline and the serving pipeline compute the same feature in two different ways.
- **Dawid-Skene**: a method that estimates each annotator's error pattern and the true labels together. **Cohen's kappa**: agreement of two annotators beyond chance. **Weak supervision**: many noisy labelling rules, combined into labels. **Active learning**: the model picks the items that a person labels next.
- **MinHash**: a short signature of a set. Two similar texts get similar signatures. Used to find near duplicates.
- **Kafka**: a log of events, split in partitions. A **consumer group** shares the partitions among its consumers. **KRaft**: Kafka's own metadata quorum, with no ZooKeeper. **At-least-once**: a crash can repeat records. **Watermark**: the stream's estimate that no older event will arrive.
- **Spark**: an engine that spreads a data job over many cores or machines. **RAPIDS Accelerator for Apache Spark**: runs Spark SQL on NVIDIA GPUs with no code change (source: `gpus-for-etl-run-faster-less-costly-workloads-with-nvidia-rapids-accelerator-for-apache-spark-and-databricks.txt`). This notebook has no GPU part, so it does not run RAPIDS or Databricks.
- **Data retention**: delete data after a set time. **Data sovereignty**: data stays in the region whose law applies to it.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Part | What you measure | Your guess |
|---|---|---|
| A | Field accuracy of the parsers for SQL rows, nested JSON and free-text emails. Which shape needs most code? | |
| B | Duplicate rows after a killed job and a rerun, with and without an idempotency key | |
| C | For each of five planted faults: found by the check? Model score before and after the fix | |
| D1 | Cross-validation R2 with the target mean on all rows against inside each fold, and the next-month R2 | |
| D2 | Best encoding of the missing pattern: drop rows, one flag, eight flags, or one integer mask | |
| E | AUC of a model trained with a naive join against a point-in-time join. Cost of training-serving skew | |
| F | Majority vote against Dawid-Skene with one adversarial annotator. Active learning against random choice | |
| G | Which text step hurts accuracy? | |
| H | Duplicates and losses after a consumer crash. Windows that stay exact when late events arrive. Does Spark beat pandas on one machine? | |
| I | Which of the three data flows has the tightest latency need? | |
| J | Does each rule test fail when the rule is off? | |

In [ ]:
# 1. Setup: imports, helpers, and background installs of the three isolated stacks (Feast, Kafka client, Spark)
import os, sys, re, json, time, math, hashlib, shutil, signal, sqlite3, subprocess, traceback, zlib, datetime, inspect, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, recall_score, r2_score, accuracy_score, cohen_kappa_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
ON_KAGGLE = os.path.isdir("/kaggle/working")
OUT = Path("/kaggle/working" if ON_KAGGLE else "./out"); OUT.mkdir(parents=True, exist_ok=True)
SLUG, T0, STATUS, CHECKS, RAW, S = "data_pipeline", time.time(), [], [], [], {}

def part(name, fn, budget=600):
    """Run one part. Print the traceback, record the status, and go on. A time budget stops a stuck part."""
    t = time.time(); print(f"\n===== PART {name} (budget {budget} s)", flush=True)
    def _stop(*a): raise TimeoutError(f"part over its budget of {budget} s")
    try: signal.signal(signal.SIGALRM, _stop); signal.alarm(budget)
    except ValueError: pass
    try: fn(); st, note = "ok", ""
    except Exception as e: traceback.print_exc(); st, note = "FAILED", f"{type(e).__name__}: {e}"[:140]
    finally:
        try: signal.alarm(0)
        except ValueError: pass
    STATUS.append(dict(part=name, status=st, seconds=round(time.time() - t, 1), note=note))

def show(name, df):
    """Print a table (stdout is the log) and save it as CSV."""
    df.to_csv(OUT / f"{SLUG}_{name}.csv", index=False); print(f"\n[{name}]"); print(df.to_string(index=False))

def wilson(k, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p, d = k / n, 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
def rate(k, n): lo, hi = wilson(k, n); return f"{k / max(n, 1):.3f} [{lo:.3f},{hi:.3f}]"
def mcnemar(a, b):
    """Exact two-sided McNemar test on two boolean arrays of the same items."""
    a, b = np.asarray(a, bool), np.asarray(b, bool); x, y = int((a & ~b).sum()), int((~a & b).sum())
    return 1.0 if x + y == 0 else float(stats.binomtest(x, x + y, 0.5).pvalue)
def selfcheck(label, check, good, bad):
    """A check must pass on a good input and fail on a bad input. Otherwise it can never fail and tells us nothing."""
    g, b = bool(check(good)), bool(check(bad)); CHECKS.append(dict(check=label, passes_on_good=g, passes_on_bad=b))
    assert g and not b, f"self-check '{label}' cannot tell good from bad: good={g} bad={b}"
def p50_95(x): return np.percentile(x, 50), np.percentile(x, 95)

selfcheck("wilson interval holds p", lambda t: t[0] < 0.5 < t[1], wilson(50, 100), wilson(90, 100))
selfcheck("mcnemar sees a real gap", lambda p: p > 0.05, mcnemar([1] * 5 + [0] * 5, [0] * 5 + [1] * 5), mcnemar([1] * 20, [0] * 20))

# Isolated stacks: one uv venv each, installed in the background while the kernel parts run.
VPY, PROCS = {}, {}
def start_venv(name, pkgs):
    d = OUT / "venvs" / name
    cmd = f"uv venv -q --python {sys.executable} {d} && uv pip install -q --python {d}/bin/python {' '.join(pkgs)}"
    PROCS[name] = subprocess.Popen(cmd, shell=True, stdout=open(OUT / f"venv_{name}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    VPY[name] = str(d / "bin" / "python")
def venv_ready(name, timeout=600):
    """Wait for the install. Return the venv python, or None when the install failed or timed out."""
    p = PROCS.get(name)
    if p is None: return None
    try: rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired: os.killpg(p.pid, signal.SIGKILL); rc = -9
    if rc: print(f"venv {name} failed (rc {rc}):", (OUT / f"venv_{name}.log").read_text()[-300:]); VPY[name] = None
    return VPY.get(name)
def run_worker(py, script_text, name, args, timeout, env=None):
    """Run a script in a venv python. Return the JSON after the line RESULT, or None. Errors are printed (stderr tail), never hidden."""
    f = OUT / name; f.write_text(script_text)
    try: r = subprocess.run([py, str(f), *map(str, args)], capture_output=True, text=True, timeout=timeout, env=env)
    except subprocess.TimeoutExpired: print(f"{name}: timeout after {timeout} s"); return None
    line = next((l for l in r.stdout.splitlines()[::-1] if l.startswith("RESULT")), None)
    if r.returncode or not line: print(f"{name}: failed (rc {r.returncode})\n" + (r.stderr or r.stdout)[-1200:]); return None
    return json.loads(line[6:])
KAFKA_DIR = OUT / "kafka"
if ON_KAGGLE:
    start_venv("feast", ["feast==0.68.0", "pandas", "pyarrow"])
    start_venv("kafka", ["kafka-python==3.0.13", "jdk4py==25.0.5.1"])
    start_venv("spark", ["pyspark==4.3.0", "jdk4py==25.0.5.1", "pandas", "pyarrow", "numpy"])
    KAFKA_DIR.mkdir(exist_ok=True)
    urls = ["https://archive.apache.org/dist/kafka/4.2.2/kafka_2.13-4.2.2.tgz", "https://downloads.apache.org/kafka/4.3.1/kafka_2.13-4.3.1.tgz"]
    PROCS["kafka_tgz"] = subprocess.Popen(" || ".join(f"(curl -sSfL --max-time 600 {u} | tar -xz -C {KAFKA_DIR})" for u in urls),
                                          shell=True, stdout=open(OUT / "kafka_download.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
else:
    print("not on Kaggle: no venvs. Feast, Kafka and Spark stacks are skipped, the Python fallbacks run.")
print("setup done; Kaggle:", ON_KAGGLE, "| kernel python", sys.version.split()[0])

## Part A: the same orders in three shapes (data.6)

The exam asks for structured, semi-structured and unstructured data. This part makes 600 orders, then writes each order three ways: as a SQL row, as a nested JSON log line, and as a free-text email. The JSON logs have schema drift: 12% rename `qty` to `quantity`, 4% lose the customer, 3% write the price as text. The emails use five templates. Two templates are hard: one uses number words, a day/month/year date and a decimal comma.

Each shape gets a parser that reads the shape and returns the six fields. The grader compares each field with the truth. A parser gets a first version and a second version, after you read where the first one failed. Outcomes are `exact` (all six fields right), `wrong` (a field has a wrong value), `partial` (a field is missing) and `unparsed` (no order ID). No row is dropped. Effort is the number of lines of the parser functions. A last table asks two questions by query only.

In [ ]:
# 2. Part A: SQL rows, nested JSON logs and free-text emails for the same 600 orders
ITEMS = {"Desk lamp": 30.0, "Keyboard": 80.0, "Monitor": 200.0, "Mouse": 15.0, "Headset": 60.0, "Webcam": 45.0}
FIELDS = ["order_id", "customer_id", "item", "qty", "unit_price", "date"]
NUMW = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8}

def make_orders(n, seed):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({"order_id": [f"O{i:05d}" for i in range(n)], "customer_id": [f"C{c:03d}" for c in r.integers(1, 80, n)],
                       "item": r.choice(list(ITEMS), n), "qty": r.integers(1, 9, n),
                       "date": (pd.Timestamp("2026-01-01") + pd.to_timedelta(r.integers(0, 300, n), "D")).strftime("%Y-%m-%d")})
    df["unit_price"] = df["item"].map(ITEMS); return df

def render_json(o, r):
    qk = "qty" if r.random() > 0.12 else "quantity"                      # schema drift: a renamed key
    order = {"id": o.order_id, "customer": {"id": o.customer_id}, "lines": [{"sku": o.item, qk: int(o.qty), "price": float(o.unit_price)}]}
    if r.random() < 0.04: del order["customer"]                           # a field that is really missing
    if r.random() < 0.03: order["lines"][0]["price"] = f"{o.unit_price:.2f}"   # a number written as text
    return json.dumps({"ts": o.date + "T10:00:00Z", "event": "order_placed", "payload": {"order": order}})

TEMPLATES = ["Hi team, please send {qty} x {item} to account {cust}. Order ref {oid}, placed {date}. Price {price:.2f} each. Thanks!",
             "Order {oid} ({date}): customer {cust} wants {qty} units of {item} at ${price:.2f} per unit.",
             "Hello, it's {cust}. I ordered {item}, quantity {qty}, on {date}. The reference is {oid}. Unit price was {price:.2f} EUR.",
             "{cust} here. Can you ship order {oid}? That is {words} {item} bought on {dmy}, {pcomma} each.",
             "Re: {oid} - {item} x{qty} for {cust} on {date} @ {price:.2f}"]
def render_email(o, k, r):
    d = o.date; words = next(w for w, v in NUMW.items() if v == o.qty)
    quote = f"> Earlier: order O{r.integers(90000, 99999)} of 2025-12-01 at 99.00 was late.\n" if r.random() < 0.2 else ""   # a quoted old thread
    return quote + TEMPLATES[k].format(oid=o.order_id, cust=o.customer_id, item=o.item, qty=o.qty, words=words, date=d, dmy=f"{d[8:]}/{d[5:7]}/{d[:4]}",
                               price=o.unit_price, pcomma=f"{o.unit_price:.2f}".replace(".", ","))

def parse_json_strict(line):                       # first version: assumes the clean schema
    try:
        d = json.loads(line); o = d["payload"]["order"]; l = o["lines"][0]
        return dict(order_id=o["id"], customer_id=o["customer"]["id"], item=l["sku"], qty=l["qty"], unit_price=l["price"], date=d["ts"][:10])
    except Exception: return None
def parse_json_tolerant(line):                     # second version: defaults, a renamed key, text numbers
    try: d = json.loads(line)
    except Exception: return None
    o = d.get("payload", {}).get("order", {}); l = (o.get("lines") or [{}])[0]; p = l.get("price")
    return dict(order_id=o.get("id"), customer_id=(o.get("customer") or {}).get("id"), item=l.get("sku"), qty=l.get("qty", l.get("quantity")),
                unit_price=float(p) if p is not None else None, date=(d.get("ts") or "")[:10] or None)

RX_ID, RX_C, RX_DATE, RX_PRICE = re.compile(r"\bO\d{5}\b"), re.compile(r"\bC\d{3}\b"), re.compile(r"\b\d{4}-\d{2}-\d{2}\b"), re.compile(r"(\d+\.\d{2})")
RX_QTY = [re.compile(p) for p in (r"(\d+) x\b", r"\bx(\d+)", r"quantity (\d+)", r"(\d+) units")]
def first(rx, t):
    m = rx.search(t); return m.group(m.lastindex or 0) if m else None
def parse_email_v1(t):
    q = next((first(rx, t) for rx in RX_QTY if rx.search(t)), None); p = first(RX_PRICE, t)
    return dict(order_id=first(RX_ID, t), customer_id=first(RX_C, t), item=next((i for i in ITEMS if i in t), None),
                qty=int(q) if q else None, unit_price=float(p) if p else None, date=first(RX_DATE, t))
def parse_email_v2(t):                             # adds number words, day/month/year dates, decimal commas
    d = parse_email_v1(t)
    if d["qty"] is None: d["qty"] = next((v for k, v in NUMW.items() if re.search(rf"\b{k}\b", t)), None)
    m = re.search(r"\b(\d{2})/(\d{2})/(\d{4})\b", t)
    if d["date"] is None and m: d["date"] = f"{m[3]}-{m[2]}-{m[1]}"
    m = re.search(r"(\d+),(\d{2})\b", t)
    if d["unit_price"] is None and m: d["unit_price"] = float(f"{m[1]}.{m[2]}")
    return d
def parse_email_v3(t):                             # third version: drop quoted lines of an old thread first
    return parse_email_v2("\n".join(l for l in t.splitlines() if not l.startswith(">")))

def same(a, b): return (isinstance(a, (int, float)) and abs(a - b) < 1e-9) if isinstance(b, float) else a == b
def grade(parsed, truth):
    """Outcome per row: exact, wrong (a value differs), partial (a field is missing), unparsed (no order ID). Also the field hits."""
    out, hits = [], []
    for p, t in zip(parsed, truth.to_dict("records")):
        if not p or not p.get("order_id"): out.append("unparsed"); hits.append([False] * 6); continue
        h = [p.get(f) is not None and same(p[f], t[f]) for f in FIELDS]; hits.append(h)
        out.append("exact" if all(h) else "partial" if any(p.get(f) is None for f in FIELDS) else "wrong")
    return out, np.array(hits)
_t = make_orders(1, 0).to_dict("records")[0]; _gp = dict(_t); _bp = dict(_t, qty=_t["qty"] + 1)
selfcheck("grader: exact vs wrong", lambda p: grade([p], pd.DataFrame([_t]))[0][0] == "exact", _gp, _bp)
selfcheck("grader: unparsed is its own outcome", lambda p: grade([p], pd.DataFrame([_t]))[0][0] != "unparsed", _gp, None)

def part_a():
    ORD = make_orders(600, 1); r = np.random.default_rng(11); S["ORD"] = ORD
    con = sqlite3.connect(":memory:"); ORD.to_sql("orders", con, index=False)
    jl = [render_json(o, r) for o in ORD.itertuples()]; ks = r.integers(0, 5, len(ORD)); em = [render_email(o, k, r) for o, k in zip(ORD.itertuples(), ks)]
    for t, k in zip(em, ks):                                   # simulator truth: the renderings contain the true values
        assert "O" in t and (k != 3 or "," in t)
    rows = []
    def run(shape, ver, fn, items):
        t = time.time(); parsed = [fn(x) for x in items]; ms = (time.time() - t) * 1000
        outc, hits = grade(parsed, ORD); c = pd.Series(outc).value_counts()
        base = np.mean([ORD[f].value_counts(normalize=True).iloc[0] for f in FIELDS])   # baseline: always the most common value of each field
        try: loc = len(inspect.getsource(fn).splitlines())
        except Exception: loc = np.nan
        rows.append(dict(shape=shape, parser=ver, exact=rate(c.get("exact", 0), len(ORD)), wrong=c.get("wrong", 0), partial=c.get("partial", 0),
                         unparsed=c.get("unparsed", 0), field_acc=rate(int(hits.sum()), hits.size), baseline=round(base, 3), lines=loc, ms=round(ms, 1)))
        return hits
    sql = lambda row: row                                      # a SQL row already has the fields
    run("structured SQL", "SELECT", sql, ORD.to_dict("records"))
    run("semi-structured JSON", "v1 strict", parse_json_strict, jl); run("semi-structured JSON", "v2 tolerant", parse_json_tolerant, jl)
    run("unstructured email", "v1 regex", parse_email_v1, em); run("unstructured email", "v2 regex+", parse_email_v2, em)
    h2 = run("unstructured email", "v3 skip quotes", parse_email_v3, em)
    show("A_parsers", pd.DataFrame(rows))
    tpl = pd.DataFrame({"template": ks, "all_fields": h2.all(1)}).groupby("template").all_fields.agg(["mean", "size"]).round(3)
    print("\nemail v3 exact rate per template:"); print(tpl.to_string())
    # What can a query do without a parser? Two questions, answered by SQL only.
    docs = sqlite3.connect(":memory:"); docs.execute("CREATE TABLE j(doc TEXT)"); docs.execute("CREATE TABLE e(body TEXT)")
    docs.executemany("INSERT INTO j VALUES (?)", [(x,) for x in jl]); docs.executemany("INSERT INTO e VALUES (?)", [(x,) for x in em])
    tr_rev = float((ORD.qty * ORD.unit_price).sum()); tr_lamp = int((ORD.item == "Desk lamp").sum()); L = "$.payload.order.lines[0]"
    q = [("structured SQL", "revenue", con.execute("SELECT SUM(qty*unit_price) FROM orders").fetchone()[0], tr_rev),
         ("structured SQL", "Desk lamp orders", con.execute("SELECT COUNT(*) FROM orders WHERE item='Desk lamp'").fetchone()[0], tr_lamp),
         ("semi-structured JSON", "revenue", docs.execute(f"SELECT SUM(json_extract(doc,'{L}.qty')*json_extract(doc,'{L}.price')) FROM j").fetchone()[0], tr_rev),
         ("semi-structured JSON", "Desk lamp orders", docs.execute(f"SELECT COUNT(*) FROM j WHERE json_extract(doc,'{L}.sku')='Desk lamp'").fetchone()[0], tr_lamp),
         ("unstructured email", "revenue", None, tr_rev),
         ("unstructured email", "Desk lamp orders", docs.execute("SELECT COUNT(*) FROM e WHERE body LIKE '%Desk lamp%'").fetchone()[0], tr_lamp)]
    qt = pd.DataFrame(q, columns=["shape", "question", "query_answer", "truth"])
    qt["error_%"] = [None if a is None else round(100 * abs(a - t) / t, 1) for a, t in zip(qt.query_answer, qt.truth)]
    show("A_queries", qt)
    RAW.append(dict(part="A", parsers=rows))
part("A three shapes of data", part_a)

## Part B: ETL with a dead-letter queue, a killed job, and ETL against ELT (data.3)

The feed holds 20,000 order events. 12% carry one planted fault: a missing customer, a zero quantity, a quantity written as text, a negative price, an impossible date or an unknown item. The source also re-sends 3% of the events with the same `event_id`.

The ETL job validates each event in Python. A valid event goes into the `orders` table. An invalid event goes into the `dlq` table with the reason. The job commits every 250 events. Then we kill the job in the middle of batch 5 and start it again from the beginning, because this job keeps no checkpoint. We run this twice. Table 1 has no key. Table 2 has `event_id` as primary key and writes with `INSERT OR IGNORE`, so a repeated write changes nothing (an idempotency key).

Checks: every distinct event ID must end up in exactly one of the two tables, and no clean event may land in the DLQ. The last step loads the raw lines first and transforms with SQL (ELT). Then a business rule changes, and we time how long each design needs to apply it.

In [ ]:
# 3. Part B: ETL, dead-letter queue, kill and rerun, ETL against ELT
BAD_KINDS = ["no_customer", "qty_zero", "qty_text", "price_negative", "bad_date", "unknown_item"]
def make_feed(n, seed):
    r = np.random.default_rng(seed); base = make_orders(n, seed)
    ev = [dict(event_id=f"E{i:06d}", order_id=o.order_id, customer_id=o.customer_id, item=o.item, qty=int(o.qty), price=float(o.unit_price), date=o.date)
          for i, o in enumerate(base.itertuples())]
    bad = {}
    for j, i in enumerate(r.choice(n, int(0.12 * n), replace=False)):
        k = BAD_KINDS[j % 6]; e = ev[i]; bad[e["event_id"]] = k
        if k == "no_customer": e["customer_id"] = None
        elif k == "qty_zero": e["qty"] = 0
        elif k == "qty_text": e["qty"] = "three"
        elif k == "price_negative": e["price"] = -5.0
        elif k == "bad_date": e["date"] = "2026-13-45"
        else: e["item"] = "Toaster"
    return ev + [dict(ev[i]) for i in r.choice(n, int(0.03 * n), replace=False)], bad

def validate(e):
    """Return the reason to reject an event, or None."""
    if not e.get("customer_id"): return "no_customer"
    if not isinstance(e.get("qty"), int): return "qty_text"
    if e["qty"] <= 0: return "qty_zero"
    if not isinstance(e.get("price"), (int, float)) or e["price"] <= 0: return "price_negative"
    if e.get("item") not in ITEMS: return "unknown_item"
    try: datetime.date.fromisoformat(e["date"])
    except Exception: return "bad_date"
    return None
selfcheck("validate: clean passes, broken fails", lambda e: validate(e) is None,
          dict(customer_id="C1", qty=2, price=3.0, item="Mouse", date="2026-01-02"), dict(customer_id="C1", qty=2, price=3.0, item="Mouse", date="2026-13-02"))

def new_db(path, keyed):
    Path(path).unlink(missing_ok=True); con = sqlite3.connect(path); pk = "PRIMARY KEY" if keyed else ""
    con.execute(f"CREATE TABLE orders(event_id TEXT {pk}, order_id, customer_id, item, qty, price, date)")
    con.execute(f"CREATE TABLE dlq(event_id TEXT {pk}, reason, raw)"); con.commit(); con.close()
def run_job(path, feed, keyed, kill_batch=None, bs=250):
    """ETL job: validate in Python, load batch by batch. A kill rolls back the open batch only. There is no checkpoint."""
    ins = "INSERT OR IGNORE" if keyed else "INSERT"; con = sqlite3.connect(path)
    for b in range(0, len(feed), bs):
        try:
            with con:                                           # one transaction per batch
                for i, e in enumerate(feed[b:b + bs]):
                    if b // bs == kill_batch and i == bs // 2: raise KeyboardInterrupt("job killed")
                    why = validate(e)
                    if why: con.execute(f"{ins} INTO dlq VALUES (?,?,?)", (e["event_id"], why, json.dumps(e)))
                    else: con.execute(f"{ins} INTO orders VALUES (?,?,?,?,?,?,?)", (e["event_id"], e["order_id"], e["customer_id"], e["item"], e["qty"], e["price"], e["date"]))
        except KeyboardInterrupt: con.close(); return "killed"
    con.close(); return "done"

def part_b():
    feed, bad = make_feed(20000, 5); ids = {e["event_id"] for e in feed}; db = str(OUT / "etl.db")
    assert len(bad) == 2400 and len(feed) == 20600 and len(ids) == 20000                       # the simulator truth
    selfcheck("every planted fault is rejected", lambda f: all(validate(e) for e in f), [e for e in feed if e["event_id"] in bad][:50], [e for e in feed if e["event_id"] not in bad][:50])
    rows = []
    for keyed in (False, True):
        for kill in (None, 4):
            new_db(db, keyed); st = run_job(db, feed, keyed, kill)
            if kill is not None: st += " then rerun " + run_job(db, feed, keyed)              # rerun from the start
            con = sqlite3.connect(db); o = [r[0] for r in con.execute("SELECT event_id FROM orders")]; d = [r[0] for r in con.execute("SELECT event_id FROM dlq")]
            reasons = dict(con.execute("SELECT event_id, reason FROM dlq")); con.close()
            hit = sum(reasons.get(k) == v for k, v in bad.items())
            rows.append(dict(idempotency_key=keyed, run=st, orders=len(o), dlq=len(d), dup_rows=len(o) + len(d) - len(set(o) | set(d)),
                             ids_lost=len(ids - set(o) - set(d)), clean_in_dlq=len(set(d) - set(bad)), dlq_reason_right=rate(hit, len(bad)), baseline=round(1 / 6, 3)))
            assert not ids - set(o) - set(d) and not set(o) & set(d), "a record was lost or sits in both tables"
    show("B_kill_and_rerun", pd.DataFrame(rows))
    assert rows[3]["dup_rows"] == 0 and rows[1]["dup_rows"] > 0 and rows[0]["clean_in_dlq"] == 0
    # ELT: load the raw lines first, then transform with SQL. ETL: transform in Python before the load.
    raw = [json.dumps(e) for e in feed]; con = sqlite3.connect(":memory:"); T = {}
    t = time.time(); con.execute("CREATE TABLE raw(doc TEXT)"); con.executemany("INSERT INTO raw VALUES (?)", [(x,) for x in raw]); T["elt_load_s"] = time.time() - t
    J = lambda k: f"json_extract(doc,'$.{k}')"
    ok_sql = (f"{J('customer_id')} IS NOT NULL AND json_type(doc,'$.qty')='integer' AND {J('qty')}>0 AND json_type(doc,'$.price') IN ('real','integer') AND {J('price')}>0 "
              f"AND {J('item')} IN ({','.join(repr(i) for i in ITEMS)}) AND date({J('date')}) IS NOT NULL")
    def elt(extra=""):
        con.execute("DROP TABLE IF EXISTS orders_elt"); t = time.time()
        con.execute(f"CREATE TABLE orders_elt AS SELECT DISTINCT {J('event_id')} AS event_id, {J('qty')} AS qty FROM raw WHERE {ok_sql} {extra}")
        return time.time() - t
    T["elt_transform_s"] = elt(); n_elt = con.execute("SELECT COUNT(*) FROM orders_elt").fetchone()[0]
    t = time.time(); etl_ok = {e["event_id"] for e in feed if validate(e) is None}; T["etl_transform_s"] = time.time() - t
    assert n_elt == len(etl_ok), (n_elt, len(etl_ok))                                           # both designs keep the same rows
    t = time.time(); etl_ok5 = {e["event_id"] for e in (json.loads(x) for x in raw) if validate(e) is None and e["qty"] <= 5}; T["etl_rule_change_s"] = time.time() - t
    T["elt_rule_change_s"] = elt("AND " + J("qty") + "<=5"); assert con.execute("SELECT COUNT(*) FROM orders_elt").fetchone()[0] == len(etl_ok5)
    show("B_etl_vs_elt", pd.DataFrame([dict(design="ETL (Python, then load)", transform_s=T["etl_transform_s"], rule_change_s=T["etl_rule_change_s"], keeps_raw_in_target=False, rows=len(etl_ok)),
                                       dict(design="ELT (load raw, then SQL)", transform_s=T["elt_transform_s"], rule_change_s=T["elt_rule_change_s"], keeps_raw_in_target=True, rows=n_elt)]).round(3))
    RAW.append(dict(part="B", kill_rerun=rows, times=T))
part("B ETL, dead-letter queue, kill and rerun", part_b)

## Part C: exploratory data analysis with five planted faults (data.1, data.4)

The table has 30,000 training rows and about 3% positive rows. Five faults are planted in the training data. The production data ("next month", 30,000 rows) is clean:

1. **Unit mix**: source system B reports `weight` in grams. The others report kilograms.
2. **Skew**: `amount` has a long right tail. The target depends on its logarithm.
3. **Missing not at random**: `income` is missing much more often when the target is 1.
4. **Imbalance**: only 3% of rows are positive.
5. **Leaking column**: `refund_flag` is set after the event, so it is 1 for almost every positive row. It is not known at prediction time, so it is 0 in production.

One check per fault looks at the data only (it never reads the planted truth). Then a logistic regression trains on the faulty data and on the fixed data. Both are scored on the clean production data. The metric is the AUC (baseline 0.5), and for the imbalance fault the recall at threshold 0.5 (baseline 0 for "always negative", which still scores 97% accuracy). Each fix is tested alone and then all five together. Each check also runs on a good and a bad input, to prove that it can fail.

In [ ]:
# 4. Part C: five faults, five checks, and the model before and after each fix

def make_eda(n, seed, production):
    r = np.random.default_rng(seed)
    src, age, w_kg = r.choice(list("ABC"), n, p=[.4, .3, .3]), r.normal(40, 12, n), r.normal(70, 12, n)
    amount, inc = np.exp(r.normal(3, 1.2, n)), r.normal(50, 15, n)
    logit = -4.1 + 0.5 * (age - 40) / 12 + 0.5 * (w_kg - 70) / 12 + 0.9 * (np.log(amount) - 3) / 1.2 + 0.4 * (inc - 50) / 15
    y = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame(dict(src=src, age=age, weight=w_kg, amount=amount, income=inc, y=y))
    df["age"] = df.age.where(r.random(n) > 0.05)                                       # ordinary random gaps, a control
    df["income"] = df.income.where(r.random(n) > np.where(y == 1, 0.6, 0.05))           # fault 3: missing depends on the target
    df["refund_flag"] = ((y == 1) & (r.random(n) < 0.97)).astype(int)                   # fault 5: known only after the event
    if production: df["refund_flag"] = 0
    else: df.loc[df.src == "B", "weight"] *= 1000                                      # fault 1: grams for system B
    return df
TR, TE = make_eda(30000, 1, False), make_eda(30000, 2, True)

# one check per fault. Each check reads the data only.
def chk_unit(d):  m = d.groupby("src").weight.median(); return m.max() / m.min() > 20
def chk_skew(x):  return abs(stats.skew(np.asarray(x, float))) > 2
def chk_mnar(d, col):                                               # does "missing" predict the target?
    f = d[col].isna().astype(int); return f.nunique() > 1 and abs(roc_auc_score(d.y, f) - 0.5) > 0.1
def chk_imbalance(y): return np.mean(y) < 0.10
def chk_leak(d, col): return roc_auc_score(d.y, d[col].fillna(0)) > 0.95
selfcheck("unit-mix check", lambda d: not chk_unit(d), TE, TR)
selfcheck("skew check", lambda x: not chk_skew(x), np.random.default_rng(0).normal(size=999), np.exp(np.random.default_rng(0).normal(size=999) * 1.2))
selfcheck("MNAR check ignores random gaps", lambda a: not chk_mnar(*a), (TR, "age"), (TR, "income"))
selfcheck("imbalance check", lambda y: not chk_imbalance(y), np.r_[np.ones(40), np.zeros(60)], TR.y)
selfcheck("leak check", lambda a: not chk_leak(*a), (TR, "income"), (TR, "refund_flag"))

def fix_train(d, fixes):
    d = d.copy()
    if "unit" in fixes: d.loc[d.src == "B", "weight"] /= 1000
    return d
def feats(d, fixes):
    X = pd.DataFrame({"age": d.age.fillna(40), "weight": d.weight, "src_A": (d.src == "A") * 1.0})
    X["amount"] = np.log(d.amount) if "skew" in fixes else d.amount
    X["income"] = d.income.fillna(50)
    if "mnar" in fixes: X["income_missing"] = d.income.isna() * 1.0
    if "leak" not in fixes: X["refund_flag"] = d.refund_flag
    return X
def fit_eval(fixes, test_set=None):
    te = TE if test_set is None else test_set
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced" if "imb" in fixes else None)).fit(feats(fix_train(TR, fixes), fixes), TR.y)
    Xt = feats(te, fixes); p = m.predict_proba(Xt)[:, 1]
    return roc_auc_score(te.y, p), recall_score(te.y, p > 0.5)

def part_c():
    found = dict(unit=chk_unit(TR), skew=chk_skew(TR.amount), mnar=chk_mnar(TR, "income"), imb=chk_imbalance(TR.y), leak=chk_leak(TR, "refund_flag"))
    stat = dict(unit=f"median weight B/A = {TR[TR.src == 'B'].weight.median() / TR[TR.src == 'A'].weight.median():.0f}", skew=f"skew of amount = {stats.skew(TR.amount):.1f}",
                mnar=f"AUC of 'income missing' vs target = {roc_auc_score(TR.y, TR.income.isna()):.2f}", imb=f"positives {TR.y.mean():.3f}, always-negative accuracy {1 - TR.y.mean():.3f}",
                leak=f"AUC of refund_flag alone = {roc_auc_score(TR.y, TR.refund_flag):.2f}")
    assert all(found.values()), found
    ctrl = make_eda(30000, 3, True); ctrl["amount"] = np.exp(np.random.default_rng(1).normal(3, .3, len(ctrl)))
    print("control (clean data): unit", chk_unit(ctrl), "| skew", chk_skew(ctrl.amount), "| mnar on random gaps", chk_mnar(ctrl, "age"), "| leak", chk_leak(ctrl, "income"))
    assert not (chk_unit(ctrl) or chk_skew(ctrl.amount) or chk_mnar(ctrl, "age") or chk_leak(ctrl, "income"))
    ALL = {"unit", "skew", "mnar", "imb", "leak"}; rows = []
    for f, label, metric in (("unit", "unit mix", "AUC"), ("skew", "skew", "AUC"), ("mnar", "missing not at random", "AUC"), ("imb", "imbalance", "recall@0.5"), ("leak", "leaking column", "AUC")):
        i = 1 if f == "imb" else 0; g = lambda fx: round(fit_eval(fx)[i], 3)
        rows.append(dict(fault=label, found_by_check=found[f], evidence=stat[f], metric=metric, all_faulty=g(set()), fix_only_this=g({f}),
                         fix_all_but_this=g(ALL - {f}), fix_all=g(ALL), baseline=0.0 if f == "imb" else 0.5))
    show("C_eda_faults", pd.DataFrame(rows))
    Xl = feats(TR, set()); m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xl, TR.y)
    print(f"\nleak: AUC on the training rows with the leaking column {roc_auc_score(TR.y, m.predict_proba(Xl)[:, 1]):.3f}, on clean production data {fit_eval(set())[0]:.3f}")
    RAW.append(dict(part="C", rows=rows))
part("C exploratory data analysis, five faults", part_c)

## Part D: feature engineering (data.1), with two questions that were answered wrong before

**D1. A target mean built on all rows.** A retail model adds `store_avg_sales`, the mean of `sales` per store, over all rows. Then it runs 5-fold cross-validation. The data has 500 stores with 6 months of history each, and one next month to predict. `sales` is a store effect plus noise. We run two pipelines. Pipeline "all rows" builds the store mean once, on all history rows, then cross-validates. Pipeline "inside the fold" builds the store mean from the training rows of each fold and applies it to the validation rows. Both pipelines then fit on all history and score the next month, with the same final model. Five seeds. A feature built from the target must be computed inside each training fold.

**D2. The missing pattern as one feature.** Eight categorical columns have gaps. The claim amount depends on which columns are missing together (a random effect for each of the 256 patterns). We compare ways to use the missing data: drop every row with a gap, impute only, one "any missing" flag, eight flags, and one integer mask: flag of column j times 2 to the power j, summed over the columns. Each pattern gets its own integer. The metric is R2 on 10,000 held-out rows, imputed the same way (baseline 0 = predict the mean).

**D3. Log and ratio features.** The target depends on the ratio of debt to income. A linear model reads raw columns, then a log, then the ratio.

In [ ]:
# 5. Part D: target mean inside the fold, the missing-pattern mask, log and ratio features

def make_sales(seed, stores=500, months=6):
    r = np.random.default_rng(seed); eff = r.normal(0, 1, stores)
    d = pd.DataFrame({"store": np.repeat(np.arange(stores), months + 1), "month": np.tile(np.arange(months + 1), stores)})
    d["promo"] = r.integers(0, 2, len(d)); d["sales"] = 10 + eff[d.store] + 0.5 * d.promo + r.normal(0, 2, len(d))
    return d[d.month < months].reset_index(drop=True), d[d.month == months].reset_index(drop=True)
def gbr(): return GradientBoostingRegressor(n_estimators=80, max_depth=3, random_state=0)
def store_mean(hist, rows): return rows.store.map(hist.groupby("store").sales.mean())
def encode(a, b, inside):
    """Store mean for the rows of a and of b. inside=False is the leak: the mean is built from all history rows, validation labels included."""
    if not inside: return store_mean(HIST, a).values, store_mean(HIST, b).values
    fa = np.full(len(a), np.nan)
    for t, v in KFold(5, shuffle=True, random_state=1).split(a): fa[v] = store_mean(a.iloc[t], a.iloc[v]).values     # out-of-fold inside the training rows
    return np.nan_to_num(fa, nan=a.sales.mean()), store_mean(a, b).fillna(a.sales.mean()).values
def fit_score(a, b, inside):
    fa, fb = encode(a, b, inside); return r2_score(b.sales, gbr().fit(np.c_[a.promo, fa], a.sales).predict(np.c_[b.promo, fb]))
def cv_r2(hist, inside):
    return np.mean([fit_score(hist.iloc[t], hist.iloc[v], inside) for t, v in KFold(5, shuffle=True, random_state=0).split(hist)])

def part_d1():
    rows = []
    for seed in range(5):
        global HIST; HIST, n = make_sales(seed); h = HIST
        rows.append(dict(seed=seed, cv_all_rows=cv_r2(h, False), cv_inside_fold=cv_r2(h, True), next_month_all_rows=fit_score(h, n, False), next_month_inside_fold=fit_score(h, n, True)))
    t = pd.DataFrame(rows); m = t.drop(columns="seed").agg(["mean", "std"]).round(3)
    show("D1_target_mean", t.round(3)); print("\nmean and sd over 5 seeds (baseline R2 of a mean predictor = 0):"); print(m.to_string())
    g_all, g_in = (t.cv_all_rows - t.next_month_all_rows).mean(), (t.cv_inside_fold - t.next_month_inside_fold).mean()
    print(f"\ngap (cross-validation minus next month): all rows {g_all:+.3f}, inside the fold {g_in:+.3f}")
    assert g_all > 0.1 and abs(g_in) < 0.08, "the leak is not visible: check the simulator"
    RAW.append(dict(part="D1", rows=rows))
part("D1 target mean on all rows", part_d1)

def make_claims(n, seed, k=8, p=0.3):
    r = np.random.default_rng(seed); eff = np.random.default_rng(7).normal(0, 1.5, 2 ** k)   # one effect table for every sample
    cats = r.integers(0, 5, (n, k)); miss = r.random((n, k)) < p; mask = (miss * (1 << np.arange(k))).sum(1)
    y = eff[mask] + 0.3 * (cats * ~miss).sum(1) / k + r.normal(0, 1, n)
    return cats, miss, mask, y
def part_d2():
    ctr, mtr, ktr, ytr = make_claims(20000, 1); cte, mte, kte, yte = make_claims(10000, 2)
    assert len(set(ktr)) == len({tuple(r) for r in mtr}) and len(set(ktr)) > 100                    # each pattern has its own integer
    mode = np.array([np.bincount(ctr[~mtr[:, j], j]).argmax() for j in range(8)])
    imp = lambda c, m: np.where(m, mode, c)
    V = {"drop rows with a gap": (lambda c, m, k: imp(c, m)), "impute only": (lambda c, m, k: imp(c, m)),
         "impute + any-missing flag": (lambda c, m, k: np.c_[imp(c, m), m.any(1)]), "impute + 8 flags": (lambda c, m, k: np.c_[imp(c, m), m]),
         "impute + integer mask": (lambda c, m, k: np.c_[imp(c, m), k])}
    rows = []
    for name, f in V.items():
        keep = ~mtr.any(1) if name.startswith("drop") else np.ones(len(ytr), bool)
        Xtr = f(ctr, mtr, ktr)[keep]; mdl = HistGradientBoostingRegressor(max_iter=200, random_state=0).fit(Xtr, ytr[keep])
        extra = {"impute + any-missing flag": 2, "impute + 8 flags": 256, "impute + integer mask": len(set(ktr))}.get(name, 1)
        rows.append(dict(encoding=name, train_rows=int(keep.sum()), patterns_told_apart=extra, test_r2=round(r2_score(yte, mdl.predict(f(cte, mte, kte))), 3), baseline_r2=0.0))
    oof = np.zeros(len(ytr))                       # a groupby feature on the mask: target mean per mask value, out-of-fold (see D1)
    for t, v in KFold(5, shuffle=True, random_state=0).split(ytr):
        oof[v] = pd.Series(ktr[v]).map(pd.Series(ytr[t]).groupby(ktr[t]).mean()).fillna(ytr[t].mean()).values
    gm = pd.Series(kte).map(pd.Series(ytr).groupby(ktr).mean()).fillna(ytr.mean()).values
    for name, A, B in (("linear: impute + 8 flags", np.c_[imp(ctr, mtr), mtr], np.c_[imp(cte, mte), mte]), ("linear: impute + mask target mean", np.c_[imp(ctr, mtr), oof], np.c_[imp(cte, mte), gm])):
        rows.append(dict(encoding=name, train_rows=len(ytr), patterns_told_apart=256 if "8 flags" in name else len(set(ktr)), test_r2=round(r2_score(yte, LinearRegression().fit(A, ytr).predict(B)), 3), baseline_r2=0.0))
    t = pd.DataFrame(rows); show("D2_missing_pattern", t); r2 = t.set_index("encoding").test_r2
    print(f"\ntree models: mask {r2['impute + integer mask']} against 8 flags {r2['impute + 8 flags']}, any-missing flag {r2['impute + any-missing flag']}, drop rows {r2['drop rows with a gap']}")
    print(f"linear models: 8 flags {r2['linear: impute + 8 flags']} against the mask target mean {r2['linear: impute + mask target mean']}. Baseline R2 = 0.0")
    RAW.append(dict(part="D2", rows=rows))
part("D2 missing pattern as one feature", part_d2)

def part_d3():
    r = np.random.default_rng(3); n = 20000; inc = np.exp(r.normal(10.5, 0.6, n)); debt = inc * np.exp(r.normal(-1.0, 0.7, n))
    y = 3 * debt / inc + r.normal(0, 0.3, n); tr, te = slice(0, 10000), slice(10000, None)
    F = {"raw (income, debt)": np.c_[inc, debt], "+ log of both": np.c_[inc, debt, np.log(inc), np.log(debt)], "+ ratio": np.c_[inc, debt, debt / inc], "log + ratio": np.c_[np.log(inc), np.log(debt), debt / inc]}
    rows = [dict(features=k, test_r2=round(r2_score(y[te], LinearRegression().fit(X[tr], y[tr]).predict(X[te])), 3), baseline_r2=0.0) for k, X in F.items()]
    show("D3_log_ratio", pd.DataFrame(rows)); RAW.append(dict(part="D3", rows=rows))
part("D3 log and ratio features", part_d3)

## Part E: a feature store (data.1)

The data is 400 customers over 90 days. Customers buy at random. Half of them churn at some day, and they buy less in the 14 days before. The feature is `spend_7d`: the money spent in the 7 days before a time. A daily snapshot table holds its value at midnight of each day (the offline store). The label is "this customer churns in the next 14 days", for 4,000 (customer, time) rows where the customer is still active.

Four measurements:

1. **Point-in-time join against a naive join.** Point-in-time takes, for each label row, the last snapshot at or before the label time (pandas `merge_asof`). The naive join takes the latest snapshot per customer, which comes after the label time. Train on the first half of the time range, test on the second half with point-in-time features, which is what serving sees. We assert that every point-in-time feature is not newer than its label.
2. **Feast.** The same join runs in Feast 0.68 (venv subprocess, local file offline store, SQLite online store). We assert that Feast gives the same values as `merge_asof`. Feast also gives the read latency of its online store.
3. **Training-serving skew.** Path A computes the snapshot with a 7-day time window. Path B is an "online" updater that sums the last 3 orders and ignores their age. We train on path A and score with path A, with path B, and with path B after the fix (both paths call one function).
4. **Offline against online read latency.** One feature row for one customer: read the snapshot file and join, against a SQLite primary-key lookup.

In [ ]:
# 6. Part E: point-in-time join, Feast, training-serving skew, read latency
START = pd.Timestamp("2026-01-01")
def simulate(n=400, days=90, seed=3):
    r = np.random.default_rng(seed); churn = np.where(r.random(n) < 0.5, r.integers(25, days + 10, n), 10 ** 6); rows = []
    for c in range(n):
        for d in range(days):
            if d >= churn[c]: break
            if r.random() < 0.35 * (0.4 if churn[c] - d <= 14 else 1.0): rows.append((c, d + r.random(), float(r.lognormal(3, .5))))
    return pd.DataFrame(rows, columns=["customer_id", "day", "amount"]), churn
def snap_window(o, n, days=90):
    """Path A: spend in the 7 days before each midnight (a time window)."""
    S = np.zeros((n, days + 1)); np.add.at(S, (o.customer_id.values, o.day.values.astype(int)), o.amount.values); C = np.cumsum(S, 1)
    rows = [(c, START + pd.Timedelta(days=d), C[c, d - 1] - (C[c, d - 8] if d >= 8 else 0)) for c in range(n) for d in range(1, days + 1)]
    return pd.DataFrame(rows, columns=["customer_id", "event_timestamp", "spend_7d"])
def snap_last3(o, n, days=90):
    """Path B: an online updater. Sum of the last 3 orders, whatever their age."""
    g = {c: x for c, x in o.groupby("customer_id")}; rows = []
    for c in range(n):
        ts, am = (g[c].day.values, g[c].amount.values) if c in g else (np.array([]), np.array([]))
        for d in range(1, days + 1): i = np.searchsorted(ts, d); rows.append((c, START + pd.Timedelta(days=d), am[max(0, i - 3):i].sum()))
    return pd.DataFrame(rows, columns=["customer_id", "event_timestamp", "spend_7d"])
def snap_shared(o, n, days=90): return snap_window(o, n, days)             # the fix: both paths call the same function

def make_labels(churn, n_rows, seed=4):
    r = np.random.default_rng(seed); c = r.integers(0, len(churn), n_rows * 3); t = r.uniform(14, 76, n_rows * 3)
    keep = churn[c] > t; c, t = c[keep][:n_rows], t[keep][:n_rows]
    return pd.DataFrame({"customer_id": c, "event_timestamp": START + pd.to_timedelta(t, "D"), "y": (churn[c] <= t + 14).astype(int)})
def ns(df, c="event_timestamp"): return df.assign(**{c: df[c].astype("datetime64[ns]")})      # one time resolution for every join
def pit(labels, snaps):
    return pd.merge_asof(ns(labels).sort_values("event_timestamp"), ns(snaps).rename(columns={"event_timestamp": "feature_ts"}).sort_values("feature_ts"),
                         left_on="event_timestamp", right_on="feature_ts", by="customer_id", direction="backward")
def latest(labels, snaps):
    return ns(labels).merge(ns(snaps).sort_values("event_timestamp").groupby("customer_id").tail(1).rename(columns={"event_timestamp": "feature_ts"}), on="customer_id")
def auc_of(model, X, y): return roc_auc_score(y, model.predict_proba(X)[:, 1])
HGB = lambda: HistGradientBoostingClassifier(max_iter=60, random_state=0)

FEAST_PY = r'''
import sys, json, time, datetime as dt, pandas as pd, numpy as np
from pathlib import Path
d = Path(sys.argv[1]); repo = d / "repo"; repo.mkdir(exist_ok=True)
fs = pd.read_csv(d / "snaps.csv", parse_dates=["event_timestamp"]); fs.to_parquet(d / "snaps.parquet")
ent = pd.read_csv(d / "entities.csv", parse_dates=["event_timestamp"])
(repo / "feature_store.yaml").write_text(f"project: nb\nregistry: {d}/registry.db\nprovider: local\nonline_store:\n  type: sqlite\n  path: {d}/online.db\n")
from feast import Entity, FeatureStore, FeatureView, Field, FileSource
from feast.types import Float64
cust = Entity(name="customer", join_keys=["customer_id"])
fv = FeatureView(name="spend", entities=[cust], ttl=dt.timedelta(days=3650), schema=[Field(name="spend_7d", dtype=Float64)],
                 source=FileSource(path=str(d / "snaps.parquet"), timestamp_field="event_timestamp"))
store = FeatureStore(repo_path=str(repo)); store.apply([cust, fv])
t = time.time(); out = store.get_historical_features(entity_df=ent, features=["spend:spend_7d"]).to_df(); hist_s = time.time() - t
out.to_csv(d / "feast_out.csv", index=False)
one = ent.iloc[:1]; hl = []
for _ in range(5): t = time.time(); store.get_historical_features(entity_df=one, features=["spend:spend_7d"]).to_df(); hl.append((time.time() - t) * 1000)
store.materialize(start_date=fs.event_timestamp.min().to_pydatetime(), end_date=fs.event_timestamp.max().to_pydatetime() + dt.timedelta(days=1))
ol = []
for i in range(300):
    t = time.time(); r = store.get_online_features(features=["spend:spend_7d"], entity_rows=[{"customer_id": int(i % 400)}]).to_dict(); ol.append((time.time() - t) * 1000)
print("RESULT" + json.dumps(dict(hist_all_s=hist_s, hist_one_ms_p50=float(np.median(hl)), online_ms_p50=float(np.percentile(ol, 50)), online_ms_p95=float(np.percentile(ol, 95)))))
'''
def part_e():
    o, churn = simulate(); A = snap_window(o, 400); B = snap_last3(o, 400); S["churn"], S["orders"], S["A"] = churn, o, A
    L = make_labels(churn, 4000); assert L.y.mean() > 0.05 and len(L) == 4000
    P, N = pit(L, A), latest(L, A)
    assert (P.feature_ts <= P.event_timestamp).all(), "point-in-time join used a newer feature"
    fut = (N.feature_ts > N.event_timestamp).mean(); direct = A.set_index(["customer_id", "event_timestamp"]).spend_7d
    chk = P.apply(lambda r: direct[(r.customer_id, r.feature_ts)], axis=1); assert np.allclose(chk, P.spend_7d)               # truth: merge_asof reads the right row
    selfcheck("point-in-time check sees a future feature", lambda df: (df.feature_ts <= df.event_timestamp).all(), P, N)
    split = START + pd.Timedelta(days=45)
    Nn = N.sort_values("event_timestamp").reset_index(drop=True); Pp = P.sort_values("event_timestamp").reset_index(drop=True)
    trn, ten = Nn.event_timestamp < split, Nn.event_timestamp >= split; trp, tep = Pp.event_timestamp < split, Pp.event_timestamp >= split

    m_naive, m_pit = HGB().fit(Nn.loc[trn, ["spend_7d"]], Nn.y[trn]), HGB().fit(Pp.loc[trp, ["spend_7d"]], Pp.y[trp])
    rows = [dict(join="naive (latest snapshot)", rows_with_future_feature=f"{fut:.2f}", train_cv_auc=round(cross_val_score(HGB(), Nn.loc[trn, ["spend_7d"]], Nn.y[trn], cv=5, scoring="roc_auc").mean(), 3),
                 next_period_auc_serving_features=round(auc_of(m_naive, Pp.loc[tep, ["spend_7d"]], Pp.y[tep]), 3)),
            dict(join="point-in-time", rows_with_future_feature="0.00", train_cv_auc=round(cross_val_score(HGB(), Pp.loc[trp, ["spend_7d"]], Pp.y[trp], cv=5, scoring="roc_auc").mean(), 3),
                 next_period_auc_serving_features=round(auc_of(m_pit, Pp.loc[tep, ["spend_7d"]], Pp.y[tep]), 3))]
    t = pd.DataFrame(rows); t["baseline_auc"] = 0.5; show("E_point_in_time", t)
    # Feast, in its own venv, on the same rows
    py = venv_ready("feast"); fr = None
    if py:
        d = OUT / "feast_work"; d.mkdir(exist_ok=True); A.to_csv(d / "snaps.csv", index=False); L[["customer_id", "event_timestamp"]].to_csv(d / "entities.csv", index=False)
        fr = run_worker(py, FEAST_PY, "feast_pit.py", [d], 900)
        if fr:
            F = pd.read_csv(d / "feast_out.csv"); F["event_timestamp"] = pd.to_datetime(F.event_timestamp, utc=True).dt.tz_localize(None); M = pit(L, A).merge(F[["customer_id", "event_timestamp", "spend_7d"]].rename(columns={"spend_7d": "feast"}), on=["customer_id", "event_timestamp"])
            ok = np.isclose(M.spend_7d.fillna(-1), M.feast.fillna(-1)); print(f"\nFeast against merge_asof: {int(ok.sum())} of {len(M)} rows equal"); assert ok.mean() > 0.999
            selfcheck("Feast equals merge_asof", lambda x: np.isclose(x[0], x[1]).all(), (M.spend_7d.fillna(-1), M.feast.fillna(-1)), (M.spend_7d.fillna(-1), M.feast.fillna(-1) + 1))
    else: print("\nFeast not run: venv missing. The pandas merge_asof numbers above stand alone.")
    # training-serving skew: model trained on path A. Score with A, with B, and with the fixed path.
    Ltr, Lte = L[L.event_timestamp < split], L[L.event_timestamp >= split]; mA = HGB().fit(pit(Ltr, A)[["spend_7d"]], pit(Ltr, A).y)
    sk = []
    for name, snaps in (("path A (training pipeline)", A), ("path B (online, last 3 orders)", B), ("path B after the fix (shared function)", snap_shared(o, 400))):
        X = pit(Lte, snaps); mism = float((np.abs(X.spend_7d.values - pit(Lte, A).spend_7d.values) > 1e-6).mean())
        sk.append(dict(serving_path=name, rows_differing=round(mism, 3), auc=round(auc_of(mA, X[["spend_7d"]], X.y), 3), baseline_auc=0.5))
    show("E_training_serving_skew", pd.DataFrame(sk)); assert sk[2]["rows_differing"] == 0 and sk[1]["rows_differing"] > 0.2
    # read latency
    con = sqlite3.connect(":memory:"); last = A.sort_values("event_timestamp").groupby("customer_id").tail(1)[["customer_id", "spend_7d"]]
    last.to_sql("online", con, index=False); con.execute("CREATE UNIQUE INDEX ix ON online(customer_id)"); A.to_csv(OUT / "offline_snaps.csv", index=False)
    on = []
    for i in range(2000): t0 = time.perf_counter(); con.execute("SELECT spend_7d FROM online WHERE customer_id=?", (i % 400,)).fetchone(); on.append((time.perf_counter() - t0) * 1000)
    off = []
    for i in range(10):
        t0 = time.perf_counter(); f = pd.read_csv(OUT / "offline_snaps.csv", parse_dates=["event_timestamp"]); x = f[(f.customer_id == i) & (f.event_timestamp <= START + pd.Timedelta(days=50))].tail(1); off.append((time.perf_counter() - t0) * 1000)
    lat = [dict(read="SQLite primary-key lookup (online store, one row)", p50_ms=round(p50_95(on)[0], 3), p95_ms=round(p50_95(on)[1], 3)),
           dict(read="read snapshot file and filter (offline, one row)", p50_ms=round(p50_95(off)[0], 3), p95_ms=round(p50_95(off)[1], 3))]
    if fr: lat += [dict(read="Feast get_online_features (one entity)", p50_ms=round(fr["online_ms_p50"], 3), p95_ms=round(fr["online_ms_p95"], 3)),
                   dict(read="Feast get_historical_features (one entity)", p50_ms=round(fr["hist_one_ms_p50"], 3), p95_ms=None)]
    show("E_read_latency", pd.DataFrame(lat)); RAW.append(dict(part="E", pit=rows, skew=sk, latency=lat, feast=fr))
part("E feature store, point-in-time, skew, latency", part_e, 900)

## Part F: labelling (data.4)

The data is 2,000 short product reviews. The true label is the sentiment of the review (1 = positive). 20% of the sentiment words point the wrong way, and a "not" flips the meaning of a word. So the task has a ceiling below 100%.

Five simulated annotators label the items. Their real accuracies are set: 0.88, 0.80, 0.72, 0.60, and one adversarial annotator at 0.30, who is wrong more often than right. Each item gets 3 of the 5 annotators. We assert that the simulated accuracies match the set ones.

Measured: pairwise Cohen's kappa. Then the label quality of the single best annotator, majority vote and Dawid-Skene (our own 15-line EM), each against the truth, and the test accuracy of a TF-IDF classifier trained on those labels. Majority vote and Dawid-Skene are compared on the same items with an exact McNemar test. Next, weak supervision: five rules that vote or abstain, with their coverage and accuracy, combined by majority vote and by an accuracy-weighted vote. The weights come from the first 200 items with gold labels, and the label quality is measured on the other 1,800 items. (Dawid-Skene does not fit here: a rule that only ever votes 0 gives it no way to tell the classes apart.) Last, active learning against random choice at the same label budget, on 8 seeds.

In [ ]:
# 7. Part F: annotators, Dawid-Skene, weak supervision, active learning
POS, NEG = ["great", "excellent", "love", "good", "fine", "amazing"], ["awful", "hate", "poor", "bad", "terrible", "boring"]
FILL = "the this a it was very really product item order shipping box price quality arrived today service seller bought works ok day week use size color".split()
def make_reviews(n, seed, noisy_style=False):
    r = np.random.default_rng(seed); texts, ys = [], []
    for _ in range(n):
        y = int(r.random() < 0.5); w = list(r.choice(FILL, r.integers(4, 9)))
        for _ in range(r.integers(1, 4)):
            p = y if r.random() < 0.8 else 1 - y                       # 20% of the sentiment words point the wrong way
            neg = r.random() < 0.25                                    # "not good" means the opposite of "good"
            word = r.choice((NEG, POS)[(1 - p) if neg else p]); w.insert(r.integers(0, len(w) + 1), ("not " + word) if neg else word)
        t = " ".join(w)
        if noisy_style: t = t.capitalize() + r.choice(["", ".", "!", "!!"]) if r.random() < 0.5 else t.upper() if r.random() < 0.1 else t
        texts.append(t); ys.append(y)
    return texts, np.array(ys)
def annotate(y, accs, k, seed):
    r = np.random.default_rng(seed); L = -np.ones((len(y), len(accs)), int)
    for i in range(len(y)):
        for j in r.choice(len(accs), k, replace=False): L[i, j] = y[i] if r.random() < accs[j] else 1 - y[i]
    return L
def majority(L, K=2):
    v = np.stack([(L == k).sum(1) for k in range(K)], 1); return np.where(v[:, 0] == v[:, 1], 1, v.argmax(1))      # a tie goes to class 1: a stated rule
def dawid_skene(L, K=2, iters=40):
    N, M = L.shape; T = np.stack([(L == k).sum(1) + 1e-2 for k in range(K)], 1); T /= T.sum(1, keepdims=True)
    for _ in range(iters):
        prior = T.mean(0); conf = np.full((M, K, K), 1e-2)                                        # conf[m, true, said]
        for m in range(M):
            for l in range(K): conf[m, :, l] += T[L[:, m] == l].sum(0)
        conf /= conf.sum(2, keepdims=True); lg = np.tile(np.log(prior), (N, 1))
        for m in range(M): ok = L[:, m] >= 0; lg[ok] += np.log(conf[m][:, L[ok, m]]).T
        T = np.exp(lg - lg.max(1, keepdims=True)); T /= T.sum(1, keepdims=True)
    return T.argmax(1), conf
def tfidf_lr(): return make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000))
selfcheck("dawid-skene recovers a flipped annotator", lambda a: a > 0.9,
          accuracy_score(np.tile([0, 1], 100), dawid_skene(np.stack([np.tile([0, 1], 100)] * 3 + [1 - np.tile([0, 1], 100)], 1))[0]), 0.5)

def part_f():
    X, y = make_reviews(2000, 21); Xte, yte = make_reviews(1000, 22); ACC = [0.88, 0.80, 0.72, 0.60, 0.30]
    L = annotate(y, ACC, 3, 5); emp = [(L[L[:, j] >= 0, j] == y[L[:, j] >= 0]).mean() for j in range(5)]
    assert np.allclose(emp, ACC, atol=0.05), emp                                                 # the simulator sets the truth
    kap = np.full((5, 5), np.nan)
    for a in range(5):
        for b in range(5):
            s = (L[:, a] >= 0) & (L[:, b] >= 0)
            if a != b and s.sum() > 50: kap[a, b] = cohen_kappa_score(L[s, a], L[s, b])
    print("empirical annotator accuracy:", np.round(emp, 3)); print("pairwise Cohen's kappa (rows and columns are annotators 0 to 4):"); print(pd.DataFrame(kap).round(2).to_string())
    ds, conf = dawid_skene(L); print("Dawid-Skene estimated accuracy per annotator:", np.round([(conf[m] * np.diag([1, 1])).sum(0) @ [0.5, 0.5] for m in range(5)], 3), "(mean of the diagonal)")
    best = int(np.argmax(emp)); lab_best = np.where(L[:, best] >= 0, L[:, best], majority(L))                 # annotator 0, and the vote when 0 did not see the item
    sets = {"single best annotator": lab_best, "majority vote": majority(L), "Dawid-Skene": ds, "gold labels": y}; rows, ok = [], {}
    for k, lab in sets.items():
        m = tfidf_lr().fit(X, lab); pr = m.predict(Xte); ok[k] = pr == yte
        rows.append(dict(labels=k, label_accuracy=rate(int((lab == y).sum()), len(y)), classifier_test_acc=rate(int(ok[k].sum()), len(yte)),
                         mcnemar_p_vs_majority=None if k == "majority vote" else round(mcnemar(ok[k], ok["majority vote"]) if "majority vote" in ok else np.nan, 4)))
    rows[0]["mcnemar_p_vs_majority"] = round(mcnemar(ok["single best annotator"], ok["majority vote"]), 4)
    t = pd.DataFrame(rows); t["baseline"] = round(max(yte.mean(), 1 - yte.mean()), 3); show("F_label_quality", t)
    print(f"label-level McNemar, Dawid-Skene against majority vote on the same 2000 items: p = {mcnemar(ds == y, majority(L) == y):.2g}")
    # weak supervision: rules that vote or abstain
    LFS = {"has a positive word": lambda t: 1 if any(w in t.split() for w in ("great", "excellent", "love", "amazing")) else -1,
           "has a negative word": lambda t: 0 if any(w in t.split() for w in ("awful", "hate", "terrible", "boring")) else -1,
           "'not good' or 'not great'": lambda t: 0 if re.search(r"not (good|great|fine)", t) else -1,
           "'not bad' or 'not awful'": lambda t: 1 if re.search(r"not (bad|awful|poor)", t) else -1,
           "contains 'fine'": lambda t: 1 if "fine" in t.split() else -1}
    W = np.array([[f(t) for f in LFS.values()] for t in X]); rows = []
    for j, k in enumerate(LFS):
        c = W[:, j] >= 0; rows.append(dict(rule=k, coverage=round(c.mean(), 3), accuracy_on_covered=rate(int((W[c, j] == y[c]).sum()), int(c.sum()))))
    show("F_weak_rules", pd.DataFrame(rows)); dev, rest = np.arange(200), np.arange(200, len(y)); cov = (W >= 0).any(1)
    acc = np.array([(W[dev][W[dev, j] >= 0, j] == y[dev][W[dev, j] >= 0]).mean() for j in range(W.shape[1])]).clip(0.05, 0.95)     # rule accuracies from 200 gold items only
    score = (np.where(W >= 0, 2 * W - 1, 0) * np.log(acc / (1 - acc))).sum(1); wv = (score > 0).astype(int); rows = []
    for k, lab in (("majority of rules", majority(W)), ("accuracy-weighted vote (weights from 200 gold items)", wv)):
        c = cov[rest]; m = tfidf_lr().fit([X[i] for i in rest if cov[i]], lab[rest][c]); a = m.predict(Xte) == yte
        rows.append(dict(combiner=k, rows_labelled=f"{cov[rest].mean():.3f}", label_accuracy_on_labelled=rate(int((lab[rest][c] == y[rest][c]).sum()), int(c.sum())), classifier_test_acc=rate(int(a.sum()), len(yte))))
    t = pd.DataFrame(rows); t["baseline"] = round(max(yte.mean(), 1 - yte.mean()), 3); show("F_weak_supervision", t)
    # active learning: same budget, uncertainty against random
    vec = TfidfVectorizer(ngram_range=(1, 2)).fit(X); P, Pt = vec.transform(X), vec.transform(Xte); res = []
    for seed in range(8):
        r = np.random.default_rng(100 + seed); init = list(r.choice(len(y), 20, replace=False)); lab = {"uncertainty": list(init), "random": list(init)}
        for budget in range(20, 201, 20):
            for k, idx in lab.items():
                m = LogisticRegression(max_iter=500).fit(P[idx], y[idx]); res.append(dict(seed=seed, method=k, budget=budget, acc=m.score(Pt, yte)))
                if budget < 200:
                    pool = np.setdiff1d(np.arange(len(y)), idx)
                    idx += list(pool[np.argsort(np.abs(m.predict_proba(P[pool])[:, 1] - 0.5))[:20]] if k == "uncertainty" else r.choice(pool, 20, replace=False))
    R = pd.DataFrame(res); piv = R.pivot_table(index="budget", columns="method", values="acc"); sd = R.pivot_table(index="budget", columns="method", values="acc", aggfunc="std")
    piv["sd_uncertainty"], piv["sd_random"] = sd.uncertainty, sd.random
    d = R[R.budget == 100].pivot(index="seed", columns="method", values="acc"); diff = (d.uncertainty - d.random).values
    boot = np.random.default_rng(0).choice(diff, (5000, len(diff))).mean(1); lo, hi = np.percentile(boot, [2.5, 97.5])
    show("F_active_learning", piv.round(3).reset_index().assign(baseline=round(max(yte.mean(), 1 - yte.mean()), 3)))
    print(f"\nat budget 100: uncertainty minus random = {diff.mean():+.3f}, paired bootstrap 95% interval [{lo:+.3f}, {hi:+.3f}] over 8 seeds")
    plt.figure(figsize=(5, 3.2)); [plt.plot(piv.index, piv[k], label=k) for k in ("uncertainty", "random")]; plt.xlabel("labels"); plt.ylabel("test accuracy"); plt.legend(); plt.tight_layout(); plt.savefig(OUT / f"{SLUG}_active_learning.png", dpi=110); plt.close()
    RAW.append(dict(part="F", annotator_acc=emp, active=res))
part("F labelling", part_f)

## Part G: text preprocessing (data.4)

The training set has 1,500 unique reviews plus 25% planted duplicates: exact copies, copies with different case and punctuation, and near copies with one changed word. The reviews also have random case and trailing punctuation. The test set has 1,000 clean, unique reviews. The classifier is TF-IDF with word unigrams and bigrams, plus logistic regression.

The pipeline adds one step at a time: normalise (lower case, strip punctuation), deduplicate with MinHash, then remove stop words with the scikit-learn list. The scikit-learn list contains "not", "no" and "never". So that step hurts reviews such as "not good". A last row removes stop words but keeps negations.

MinHash uses 5-character shingles, 64 hash functions and 16 bands of 4 rows. A pair counts as a duplicate when the exact Jaccard similarity of the shingle sets is at least a threshold. We choose the threshold on one half of the data (F1 against the planted duplicates) and measure precision and recall on the other half. Each step reports tokens per review, vocabulary size and test accuracy, with an exact McNemar test against the step before.

In [ ]:
# 8. Part G: normalise, deduplicate with MinHash, remove stop words
def norm(t): return " ".join(re.sub(r"[^a-z0-9 ]", " ", t.lower()).split())
def shingles(s, k=5): s = f" {s} "; return {zlib.crc32(s[i:i + k].encode()) % (2 ** 31 - 1) for i in range(len(s) - k + 1)}
def minhash(sets, perms=64, seed=0):
    r = np.random.default_rng(seed); a, b = r.integers(1, 2 ** 31 - 1, perms, dtype=np.int64), r.integers(0, 2 ** 31 - 1, perms, dtype=np.int64)
    return np.stack([((a * np.fromiter(s, np.int64)[:, None] + b) % (2 ** 31 - 1)).min(0) for s in sets])
def dup_rows(texts, thr, bands=16):
    """Index of every row that is a near duplicate of an earlier row (MinHash buckets, then an exact Jaccard check)."""
    sh = [shingles(norm(t)) for t in texts]; sig = minhash(sh); buckets, keys = {}, []
    for i, s in enumerate(sig):
        keys.append([(b, tuple(s[b * 4:b * 4 + 4])) for b in range(bands)])
        for k in keys[-1]: buckets.setdefault(k, []).append(i)
    removed = set()
    for i in range(len(texts)):
        if i in removed: continue
        cand = {j for k in keys[i] for j in buckets[k] if j > i}
        removed |= {j for j in cand if len(sh[i] & sh[j]) / len(sh[i] | sh[j]) >= thr}
    return removed
def plant_dups(texts, y, seed):
    r = np.random.default_rng(seed); T, Y, G = list(texts), list(y), list(range(len(texts)))
    for i in r.choice(len(texts), len(texts) // 4, replace=False):
        k = r.integers(0, 3); t = texts[i]
        t = t if k == 0 else (t.upper() + "!!") if k == 1 else " ".join(w if j != 2 else "really" for j, w in enumerate(t.split()))
        T.append(t); Y.append(y[i]); G.append(i)
    p = r.permutation(len(T)); return [T[i] for i in p], np.array(Y)[p], np.array(G)[p]
def prf(pred, truth, n):
    p, t = set(pred), set(truth); tp = len(p & t); pr, rc = tp / max(len(p), 1), tp / max(len(t), 1); return pr, rc, 2 * pr * rc / max(pr + rc, 1e-9)
selfcheck("minhash dedup finds a copy and keeps different texts", lambda r: r == {1}, dup_rows(["not good at all, the product", "Not good at all, the product!", "great seller really fast"], 0.6), {0})
selfcheck("stop-word list contains 'not'", lambda w: "not" in w, ENGLISH_STOP_WORDS, {"the"})

def part_g():
    X0, y0 = make_reviews(1500, 31, noisy_style=True); X, y, G = plant_dups(X0, y0, 32); Xte, yte = make_reviews(1000, 33, noisy_style=True)
    seen, truth = set(), set()
    for i, g in enumerate(G): truth.add(i) if g in seen else seen.add(g)                      # a row is a duplicate when its group appeared earlier
    half = G % 2 == 0; A, Bm = np.flatnonzero(half), np.flatnonzero(~half)         # groups never cross halves
    def f1_at(ids, thr):
        sub = [X[i] for i in ids]; pred = {ids[j] for j in dup_rows(sub, thr)}; tr = truth & set(ids)
        return prf(pred, tr, len(ids))
    grid = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]; dev = {t: f1_at(list(A), t) for t in grid}; thr = max(grid, key=lambda t: dev[t][2])
    pr, rc, f1 = f1_at(list(Bm), thr)
    show("G_minhash", pd.DataFrame([dict(threshold=t, dev_precision=round(v[0], 3), dev_recall=round(v[1], 3), dev_f1=round(v[2], 3), chosen=(t == thr)) for t, v in dev.items()]))
    print(f"\nchosen threshold {thr} on half A; on the other half B: precision {pr:.3f}, recall {rc:.3f}, F1 {f1:.3f} (planted duplicates: {len(truth)} of {len(X)} rows)")
    dups = dup_rows(X, thr); keep = np.array([i for i in range(len(X)) if i not in dups]); SW = set(ENGLISH_STOP_WORDS); NEGS = {"not", "no", "nor", "never", "cannot"}
    sw = lambda t, drop: " ".join(w for w in t.split() if w not in drop)
    steps = [("raw text, split on spaces", list(range(len(X))), lambda t: t, lambda t: t), ("+ normalise", list(range(len(X))), norm, norm),
             ("+ MinHash dedup", keep, norm, norm), ("+ stop words (sklearn list)", keep, lambda t: sw(norm(t), SW), lambda t: sw(norm(t), SW)),
             ("+ stop words, keep negations", keep, lambda t: sw(norm(t), SW - NEGS), lambda t: sw(norm(t), SW - NEGS))]
    rows, prev = [], None
    for name, ids, f, g in steps:
        tr = [f(X[i]) for i in ids]; te = [g(t) for t in Xte]
        vec = CountVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None, ngram_range=(1, 2)); m = LogisticRegression(max_iter=2000).fit(vec.fit_transform(tr), y[ids])
        ok = m.predict(vec.transform(te)) == yte
        rows.append(dict(step=name, train_docs=len(ids), tokens_per_doc=round(np.mean([len(t.split()) for t in tr]), 2), vocab=len(vec.vocabulary_),
                         test_acc=rate(int(ok.sum()), len(yte)), baseline=round(max(yte.mean(), 1 - yte.mean()), 3), mcnemar_p_vs_previous=None if prev is None else round(mcnemar(ok, prev), 4))); prev = ok
    t = pd.DataFrame(rows); show("G_pipeline", t)
    ex = "this was not good"; print(f"\nexample: '{ex}' -> sklearn stop words -> '{sw(ex, SW)}' ; keep negations -> '{sw(ex, SW - NEGS)}'")
    RAW.append(dict(part="G", rows=rows, minhash=dict(thr=thr, precision=pr, recall=rc)))
part("G text preprocessing", part_g)

## Part H: batch against streaming (data.10, data.3)

**H1. A Kafka log, consumer groups, a crash and a resume.** The notebook downloads Apache Kafka, formats a single-node KRaft broker with the Java runtime from `jdk4py` and starts it. The limit is 5 minutes. If the broker does not start, the same experiment runs on an in-memory fake with the same calls, and the part says so. The experiment runs on the fake every time, so you can compare the two. A producer writes 2,000 orders keyed by customer into 4 partitions. We measure the records per partition and that one key always lands in one partition. Two consumers in one group share the partitions. Then a consumer crashes in the middle of a batch (after 550 records) and a new consumer resumes from the committed offset. We count what each sink holds:

- **at-least-once** (process, then commit): the crash repeats records. A sink that appends has duplicates. A sink that writes by order ID (an idempotent sink) has none.
- **at-most-once** (commit, then process): the crash loses records.

**H2. Event-time windows with late events.** 30,000 events over 6 hours. 15% arrive late, 1 to 30 minutes after their event time. Windows are 5 minutes of event time. The truth is a batch recompute over all events. A streaming job emits a window when the watermark (largest event time seen minus an allowed lateness) passes the window end. An event that arrives after its window was emitted is dropped, and we count it. We choose the allowed lateness on the first 3 hours (the smallest value with at least 99% exact windows) and measure it on the last 3 hours. The batch job runs every hour.

**H3. Spark in local mode against pandas.** `pyspark` 4.3.0 runs in a venv with the Java runtime from `jdk4py`. The job is a join with a small table, a group-by with a sum and a count, at 100 thousand, 2 million and 20 million rows. Both engines start from data in memory. Spark's input is cached before the timer starts. The check sum of both results must match. Spark starts one process and uses 4 cores here. It does not use a cluster, so this measures its overhead and not its scale-out.

In [ ]:
# 9. Part H1: Kafka (KRaft) broker, one worker script for the real client and an in-memory fake
STREAM_PY = r'''
import sys, json, time, zlib
mode, bootstrap, topic = sys.argv[1], sys.argv[2], sys.argv[3]
N, PARTS = 2000, 4
class FakeBus:
    """In-memory topic with the same calls as KafkaBus."""
    def __init__(s): s.log = [[] for _ in range(PARTS)]; s.committed, s.members, s.sent = {}, {}, []
    def send(s, key, val): p = zlib.crc32(key.encode()) % PARTS; s.log[p].append((key, val)); s.sent.append(p)
    def flush(s): return s.sent
    def consumer(s, group): return FakeConsumer(s, group)
class FakeConsumer:
    def __init__(s, bus, group): s.bus, s.group = bus, group; s.members = bus.members.setdefault(group, []); s.members.append(s); s.pos = {}; bus.committed.setdefault(group, {})
    def assignment(s): i = s.members.index(s); return [p for p in range(PARTS) if p % len(s.members) == i]
    def poll(s, n, wait=0):
        out = []
        for p in s.assignment():
            while len(out) < n and s.pos.setdefault(p, s.bus.committed[s.group].get(p, 0)) < len(s.bus.log[p]):
                o = s.pos[p]; k, v = s.bus.log[p][o]; out.append((p, o, k, v)); s.pos[p] = o + 1
        return out
    def commit(s): s.bus.committed[s.group].update(s.pos)
    def close(s): s.members.remove(s)
class KafkaBus:
    def __init__(s):
        from kafka import KafkaProducer
        s.p = KafkaProducer(bootstrap_servers=bootstrap, key_serializer=str.encode, value_serializer=str.encode, linger_ms=5); s.futs = []
    def send(s, key, val): s.futs.append(s.p.send(topic, key=key, value=val))
    def flush(s): s.p.flush(timeout=60); return [f.get(timeout=30).partition for f in s.futs]
    def consumer(s, group): return KafkaCons(group)
class KafkaCons:
    def __init__(s, group):
        from kafka import KafkaConsumer
        s.first = True; s.c = KafkaConsumer(topic, bootstrap_servers=bootstrap, group_id=group, enable_auto_commit=False, auto_offset_reset="earliest",
                                            key_deserializer=bytes.decode, value_deserializer=bytes.decode, session_timeout_ms=10000)
    def assignment(s): return sorted(tp.partition for tp in (s.c.assignment() or ()))
    def poll(s, n, wait=3):
        out, end = [], time.time() + (max(wait, 20) if s.first else wait)
        while not out and time.time() < end:
            for tp, recs in s.c.poll(timeout_ms=500, max_records=n).items(): out += [(tp.partition, r.offset, r.key, r.value) for r in recs]
        s.first = False; return out
    def commit(s): s.c.commit()
    def close(s):
        try: s.c.close(autocommit=False)
        except TypeError: s.c.close()
def run(c, sinks, crash_after, commit_first, bs=100):
    seen = 0
    while True:
        recs = c.poll(bs)
        if not recs: return "drained"
        if commit_first: c.commit()
        for p, o, k, v in recs:
            oid = json.loads(v)["order_id"]; sinks["append"].append(oid); sinks["upsert"][oid] = 1; seen += 1
            if crash_after is not None and seen == crash_after: return "crashed"
        if not commit_first: c.commit()
def delivery(bus, tag):
    rows = []
    for name, cf in (("at-least-once", False), ("at-most-once", True)):
        g = f"g-{tag}-{int(cf)}"; sinks = {"append": [], "upsert": {}}
        c1 = bus.consumer(g); s1 = run(c1, sinks, 550, cf); c1.close(); c2 = bus.consumer(g); s2 = run(c2, sinks, None, cf); c2.close()
        a = sinks["append"]; u = sinks["upsert"]
        rows.append(dict(semantics=name, run1=s1, run2=s2, append_sink_rows=len(a), append_sink_duplicates=len(a) - len(set(a)), append_sink_lost=N - len(set(a)),
                         upsert_sink_rows=len(u), upsert_sink_duplicates=0, upsert_sink_lost=N - len(u)))
    return rows
def group_test(bus, tag):
    g = f"grp-{tag}"; c1, c2 = bus.consumer(g), bus.consumer(g); seen = [set(), set()]; t = time.time()
    while time.time() - t < 45:
        for k, c in enumerate((c1, c2)):
            seen[k] |= {(p, o) for p, o, _, _ in c.poll(200, 1)}
            try: c.commit()
            except Exception: pass                    # a commit can fail while the group rebalances
        a1, a2 = set(c1.assignment()), set(c2.assignment())
        if a1 and a2 and not a1 & a2 and a1 | a2 == set(range(PARTS)) and len(seen[0] | seen[1]) == N: break
    c1.close(); c2.close()
    return dict(consumer1_partitions=sorted(a1), consumer2_partitions=sorted(a2), records1=len(seen[0]), records2=len(seen[1]), distinct=len(seen[0] | seen[1]), read_twice=len(seen[0] & seen[1]))
res = {"backend": mode}
try:
    bus = FakeBus() if mode == "fake" else KafkaBus(); keys = [f"C{i % 50:03d}" for i in range(N)]
    for i, k in enumerate(keys): bus.send(k, json.dumps({"order_id": f"O{i:05d}", "amount": 10 + i % 7}))
    parts = bus.flush(); res["partition_counts"] = [parts.count(p) for p in range(PARTS)]
    res["key_lands_in_one_partition"] = all(len({parts[i] for i, kk in enumerate(keys) if kk == k}) == 1 for k in set(keys))
    res["delivery"] = delivery(bus, "x")
    try: res["group"] = group_test(bus, "x")
    except Exception as e: res["group_error"] = repr(e)[:300]
except Exception as e: res["error"] = repr(e)[:600]
print("RESULT" + json.dumps(res))
'''
KRAFT = """process.roles=broker,controller
node.id=1
controller.quorum.voters=1@localhost:9093
listeners=PLAINTEXT://localhost:9092,CONTROLLER://localhost:9093
advertised.listeners=PLAINTEXT://localhost:9092
inter.broker.listener.name=PLAINTEXT
controller.listener.names=CONTROLLER
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
log.dirs={data}
num.partitions=4
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
group.initial.rebalance.delay.ms=0
"""
def start_broker(limit=300):
    """Format and start a one-node KRaft broker. Return (process, '') or (None, reason). The 5-minute limit counts from the start of the format step."""
    import socket
    py = venv_ready("kafka")
    if not py: return None, "no kafka venv (install failed or not on Kaggle)"
    try: PROCS["kafka_tgz"].wait(timeout=600)
    except subprocess.TimeoutExpired: return None, "kafka download timed out"
    homes = sorted(KAFKA_DIR.glob("kafka_*"))
    if not homes: return None, "no kafka download: " + (OUT / "kafka_download.log").read_text()[-200:]
    t0, home = time.time(), homes[0]
    jh = subprocess.check_output([py, "-c", "import jdk4py; print(jdk4py.JAVA_HOME)"], text=True, timeout=120).strip()
    env = dict(os.environ, JAVA_HOME=jh, KAFKA_HEAP_OPTS="-Xmx1g -Xms256m"); data = OUT / "kraft-data"; shutil.rmtree(data, ignore_errors=True)
    conf = OUT / "kraft.properties"; conf.write_text(KRAFT.format(data=data)); sh = lambda n, *a: subprocess.run([str(home / "bin" / n), *a], env=env, capture_output=True, text=True, timeout=120)
    cid = sh("kafka-storage.sh", "random-uuid").stdout.strip(); f = sh("kafka-storage.sh", "format", "-t", cid, "-c", str(conf))
    if f.returncode or not cid: return None, "kafka-storage format failed: " + (f.stderr or f.stdout)[-400:]
    log = OUT / "kafka_server.log"; proc = subprocess.Popen([str(home / "bin" / "kafka-server-start.sh"), str(conf)], env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    while time.time() - t0 < limit:
        if proc.poll() is not None: return None, "broker exited: " + log.read_text()[-500:]
        try: socket.create_connection(("localhost", 9092), 1).close(); time.sleep(3); return proc, ""
        except OSError: time.sleep(2)
    os.killpg(proc.pid, signal.SIGKILL); return None, f"broker not ready after {limit} s: " + log.read_text()[-300:]

def part_h1():
    S["broker"], why = start_broker(300)
    print("real Kafka broker:", "running (KRaft, jdk4py)" if S["broker"] else f"NOT running: {why}\n  -> the in-memory fake carries the experiment")
    runs = [("fake", ["fake", "-", "t"], sys.executable)]
    if S["broker"]: runs.append(("kafka", ["kafka", "localhost:9092", f"orders-{int(time.time())}"], VPY["kafka"]))
    drows, grows, prow = [], [], []
    for name, args, py in runs:
        r = run_worker(py, STREAM_PY, "h_stream.py", args, 400)
        if not r or "error" in r: print(name, "backend error:", (r or {}).get("error", "no result")); continue
        prow.append(dict(backend=name, records_per_partition=r["partition_counts"], key_lands_in_one_partition=r["key_lands_in_one_partition"]))
        drows += [dict(backend=name, **d) for d in r["delivery"]]
        grows.append(dict(backend=name, **r["group"]) if "group" in r else dict(backend=name, error=r["group_error"]))
    assert prow and sum(prow[0]["records_per_partition"]) == 2000 and prow[0]["key_lands_in_one_partition"]
    show("H1_partitions", pd.DataFrame(prow)); show("H1_consumer_group", pd.DataFrame(grows)); show("H1_crash_and_resume", pd.DataFrame(drows))
    for d in drows:
        assert (d["semantics"] == "at-least-once") == (d["append_sink_duplicates"] > 0) and d["upsert_sink_duplicates"] == 0 and (d["semantics"] == "at-most-once") == (d["append_sink_lost"] > 0), d
    RAW.append(dict(part="H1", partitions=prow, delivery=drows, group=grows))
part("H1 Kafka partitions, consumer group, crash and resume", part_h1, 1000)

# 10. Part H2: event-time windows, watermark against hourly batch
def make_events(n, seed, horizon=6 * 3600):
    r = np.random.default_rng(seed); t = np.sort(r.uniform(0, horizon, n)); late = r.random(n) < 0.15
    return pd.DataFrame(dict(event_time=t, arrival=t + np.where(late, r.uniform(60, 1800, n), r.exponential(5, n)), late=late))
def watermark_stream(ev, lateness, size=300):
    """Emit a window when the watermark passes its end. Return one row per window and the events with a 'dropped' flag."""
    ev = ev.sort_values("arrival").reset_index(drop=True); t, a = ev.event_time.values, ev.arrival.values
    wm = np.maximum.accumulate(t) - lateness; ev["dropped"] = np.r_[-np.inf, wm[:-1]] >= (t // size + 1) * size       # its window was already emitted when it arrived
    kept = ev[~ev.dropped]; got = kept.groupby((kept.event_time // size).astype(int)).size()
    truth = ev.groupby((ev.event_time // size).astype(int)).size(); w = pd.DataFrame({"truth": truth, "got": got}).fillna(0)
    w["fresh_s"] = a[np.minimum(np.searchsorted(wm, (w.index.values + 1) * size, side="left"), len(a) - 1)] - (w.index.values + 1) * size
    return w, ev
def batch_hourly(ev, size=300):
    ev = ev.assign(w=(ev.event_time // size).astype(int)); last = ev.groupby("w").arrival.max()                      # the first hourly run after the last arrival of the window
    return pd.DataFrame({"truth": ev.groupby("w").size(), "got": ev.groupby("w").size(), "fresh_s": np.ceil(last / 3600) * 3600 - (last.index.values + 1) * size})
def part_h2():
    ev = make_events(30000, 7); assert abs(ev.late.mean() - 0.15) < 0.01                                          # the simulator truth
    selfcheck("watermark: a long lateness drops nothing, none drops events", lambda L: watermark_stream(ev, L)[1].dropped.sum() == 0, 1800, 0)
    rows, wins = [], {}
    for L in (0, 60, 300, 900, 1800):
        w, e = watermark_stream(ev, L); wins[L] = (w, e)
    wb = batch_hourly(ev); half = lambda w, e, h: (w[(w.index < 36) == h], e[(e.event_time < 3 * 3600) == h])
    for name, L in [(f"streaming, allowed lateness {L} s", L) for L in wins] + [("hourly batch recompute", None)]:
        w, e = (wb, ev.assign(dropped=False)) if L is None else wins[L]
        for hname, h in (("first 3 h", True), ("last 3 h", False)):
            ww, ee = half(w, e, h); ex = int((ww.got == ww.truth).sum())
            rows.append(dict(method=name, half=hname, windows_exact=rate(ex, len(ww)), events_dropped=rate(int(ee.dropped.sum()), len(ee)), fresh_mean_s=round(ww.fresh_s.mean()), fresh_p95_s=round(ww.fresh_s.quantile(.95)), _ex=ex / len(ww)))
    t = pd.DataFrame(rows); chosen = min((L for L in wins if t[(t.method == f"streaming, allowed lateness {L} s") & (t.half == "first 3 h")]._ex.iloc[0] >= 0.99), default=1800)
    show("H2_windows", t.drop(columns="_ex")); print(f"\nlateness chosen on the first 3 hours (smallest with at least 99% exact windows): {chosen} s. Read the 'last 3 h' rows of that method for the measured result.")
    batch_exact = (wb.got == wb.truth).all(); assert batch_exact
    plt.figure(figsize=(5, 3.2)); s = t[t.half == "last 3 h"]; plt.scatter(s.fresh_mean_s, s._ex * 100)
    for _, r in s.iterrows(): plt.annotate(r.method.replace("streaming, allowed lateness ", "L=").replace(" recompute", ""), (r.fresh_mean_s, r._ex * 100), fontsize=7)
    plt.xlabel("mean seconds after window end"); plt.ylabel("exact windows %"); plt.tight_layout(); plt.savefig(OUT / f"{SLUG}_freshness_vs_correctness.png", dpi=110); plt.close()
    RAW.append(dict(part="H2", rows=t.drop(columns="_ex").to_dict("records"), chosen_lateness=chosen))
part("H2 event-time windows and late events", part_h2)

# 11. Part H3: Spark local mode against pandas
PD_SRC = '''
import numpy as np, pandas as pd, time
def make(n):
    rng = np.random.default_rng(0); return pd.DataFrame({"k": rng.integers(0, 1000, n), "v": rng.random(n)}), pd.DataFrame({"k": np.arange(1000), "w": rng.random(1000)})
def pandas_job(df, dim): return df.merge(dim, on="k").assign(x=lambda d: d.v * d.w).groupby("k").x.agg(["sum", "count"]).reset_index()
'''
SPARK_PY = PD_SRC + r'''
import sys, json
sizes = [int(x) for x in sys.argv[1].split(",")]; res = []; startup = err = None
try:
    t = time.time()
    from pyspark.sql import SparkSession, functions as F
    spark = SparkSession.builder.master("local[4]").appName("nb").config("spark.ui.enabled", "false").config("spark.driver.memory", "8g").config("spark.sql.shuffle.partitions", "8").getOrCreate()
    spark.range(10).count(); startup = time.time() - t
except Exception as e: spark = None; err = repr(e)[:400]
for n in sizes:
    df, dim = make(n); t = time.time(); out = pandas_job(df, dim); row = dict(rows=n, pandas_s=time.time() - t, pandas_check=float(out["sum"].sum()), pandas_count=int(out["count"].sum()))
    if spark:
        try:
            sdf, sdim = spark.createDataFrame(df).cache(), spark.createDataFrame(dim); sdf.count()
            t = time.time(); r = sdf.join(sdim, "k").groupBy("k").agg(F.sum(F.col("v") * F.col("w")).alias("s"), F.count("*").alias("c")).collect()
            row.update(spark_s=time.time() - t, spark_check=float(sum(x["s"] for x in r)), spark_count=int(sum(x["c"] for x in r))); sdf.unpersist()
        except Exception as e: row["spark_error"] = repr(e)[:300]
    res.append(row)
print("RESULT" + json.dumps(dict(startup_s=startup, error=err, rows=res)))
'''
def part_h3():
    sizes = [100_000, 2_000_000, 20_000_000]; py, r = venv_ready("spark"), None
    if py:
        jh = subprocess.check_output([py, "-c", "import jdk4py; print(jdk4py.JAVA_HOME)"], text=True, timeout=120).strip()
        r = run_worker(py, SPARK_PY, "spark_vs_pandas.py", [",".join(map(str, sizes))], 900, env=dict(os.environ, JAVA_HOME=jh))
    if not r or r.get("error"):
        print("Spark did not run:", (r or {}).get("error", "no venv or worker failed (see above)"), "\n  -> pandas alone, Spark rows are 'not reached'")
        g = {}; exec(PD_SRC, g); rows = []
        for n in sizes: df, dim = g["make"](n); t = time.time(); g["pandas_job"](df, dim); rows.append(dict(rows=n, pandas_s=time.time() - t))
        r = dict(startup_s=None, rows=rows)
    t = pd.DataFrame(r["rows"]); t["spark_s"] = t.get("spark_s", np.nan); t["spark"] = np.where(t.spark_s.isna(), "not reached", "ran"); t["speedup_spark_over_pandas"] = (t.pandas_s / t.spark_s).round(2)
    if "spark_check" in t:
        ok = np.isclose(t.spark_check, t.pandas_check, rtol=1e-6) & (t.spark_count == t.pandas_count); t["same_result"] = ok; assert ok.all(), "Spark and pandas disagree"
    show("H3_spark_vs_pandas", t.drop(columns=[c for c in ("pandas_check", "spark_check", "pandas_count", "spark_count") if c in t]).round(3)); print("Spark startup seconds:", r.get("startup_s"))
    RAW.append(dict(part="H3", result=r))
part("H3 Spark local against pandas", part_h3, 900)

## Part I: three data flows through one small system (data.10)

One small system predicts churn and answers policy questions. It has a table of 3,000 customers, a model, an online store and a search index over 500 policy chunks. The same system has three data flows:

- **Training flow**: read the table, fit the model, write the model file. A batch job.
- **Inference flow**: a request arrives, the system reads the customer's features from the online store, the model predicts, a response goes out.
- **Retrieval flow**: a question arrives, the system finds the 3 best chunks in the index. The index was built earlier by a batch job.

For each flow the code measures the bytes that move and the latency (500 requests for inference, 200 questions for retrieval). Frequency and the latency each flow needs are stated assumptions, marked as such. The "meets" column compares the measured p95 with the stated need. A retrieval check asserts that the right chunk is in the top 3 (baseline: 3 of 500 by chance).

In [ ]:
# 12. Part I: training, inference and retrieval data flows
import pickle
def part_i():
    r = np.random.default_rng(9); n = 3000; tab = pd.DataFrame({"customer_id": np.arange(n), "tenure": r.integers(1, 60, n), "spend_30d": r.lognormal(3, .6, n)})
    tab["churn"] = ((tab.tenure < 12) ^ (r.random(n) < .2)).astype(int); path = OUT / "train_table.csv"; tab.to_csv(path, index=False); rows = []
    t0 = time.perf_counter(); df = pd.read_csv(path); model = HistGradientBoostingClassifier(max_iter=60, random_state=0).fit(df[["tenure", "spend_30d"]], df.churn); blob = pickle.dumps(model); train_ms = (time.perf_counter() - t0) * 1000
    rows.append(dict(flow="training", style="batch", moves=f"{n} rows in, model file out", bytes_per_run=os.path.getsize(path) + len(blob), how_often="nightly (assumed)", p50_ms=round(train_ms), p95_ms=round(train_ms), need_ms="3600000 (assumed)", meets=train_ms < 3.6e6))
    con = sqlite3.connect(":memory:"); tab[["customer_id", "tenure", "spend_30d"]].to_sql("online", con, index=False); con.execute("CREATE UNIQUE INDEX i ON online(customer_id)"); lat, size = [], []
    for i in range(500):
        t0 = time.perf_counter(); req = json.dumps({"customer_id": int(r.integers(0, n))}); cid = json.loads(req)["customer_id"]
        feats = con.execute("SELECT tenure, spend_30d FROM online WHERE customer_id=?", (cid,)).fetchone(); p = model.predict_proba(np.array([feats]))[0, 1]
        resp = json.dumps({"customer_id": cid, "churn_probability": float(p)}); lat.append((time.perf_counter() - t0) * 1000); size.append(len(req) + len(resp))
    a, b = p50_95(lat); rows.append(dict(flow="inference", style="online lookup", moves="one request, one feature row, one score", bytes_per_run=int(np.mean(size)), how_often="every request", p50_ms=round(a, 2), p95_ms=round(b, 2), need_ms="100 (assumed)", meets=b < 100))
    topics = ["refund", "shipping", "warranty", "privacy", "returns", "billing"]; docs = [f"policy {i} about {topics[i % 6]}: customers may ask about {topics[i % 6]} for item{i} within {5 + i % 30} days" for i in range(500)]
    t0 = time.perf_counter(); vec = TfidfVectorizer().fit(docs); M = vec.transform(docs); build_ms = (time.perf_counter() - t0) * 1000; lat, hit, size = [], 0, []
    for j in r.integers(0, 500, 200):
        t0 = time.perf_counter(); q = f"what is the {topics[j % 6]} rule for item{j}"; top = np.argsort(-(M @ vec.transform([q]).T).toarray().ravel())[:3]; ctx = [docs[i] for i in top]
        lat.append((time.perf_counter() - t0) * 1000); hit += int(j in top); size.append(len(q) + sum(map(len, ctx)))
    assert hit / 200 > 0.95, hit
    a, b = p50_95(lat); rows.append(dict(flow="retrieval (query)", style="online search", moves="one question in, 3 chunks out", bytes_per_run=int(np.mean(size)), how_often="every question", p50_ms=round(a, 2), p95_ms=round(b, 2), need_ms="300 (assumed)", meets=b < 300))
    rows.append(dict(flow="retrieval (index build)", style="batch", moves=f"500 chunks in, index out ({M.nnz} entries)", bytes_per_run=M.nnz * 12, how_often="hourly (assumed)", p50_ms=round(build_ms, 1), p95_ms=round(build_ms, 1), need_ms="3600000 (assumed)", meets=build_ms < 3.6e6))
    show("I_three_flows", pd.DataFrame(rows)); print(f"retrieval hit@3: {rate(hit, 200)} (baseline by chance {3 / 500:.3f})")
    RAW.append(dict(part="I", rows=rows))
part("I three data flows", part_i)

## Part J: rules on data (data.5)

Each rule is code. Each rule has a check that reads the final state, not the code. Each check must pass when the rule is on and fail when the rule is off. The table proves that. PII redaction is not repeated here (see `data-flywheel-cudf.ipynb`).

- **Retention**: delete events older than 90 days, also in the derived `features` table. The proof is a check of the oldest row in every table, plus a deletion certificate (cutoff, number of rows, SHA-256 of the sorted IDs). Rule off means no purge. A second case purges only the first table.
- **Sovereignty**: a record that is not flagged "US" never goes to the global store. 10% of the records have a bad flag (missing, or "eu " with a space). A missing flag is treated as EU (default deny). The cost is that some US records are blocked too, and the table counts them.
- **Bias**: a model trained on past decisions that were unfair to group 1. The protected column is not a feature, but a ZIP-code proxy is. The check is the selection-rate ratio of group 1 over group 0, which must be at least 0.8 (the four-fifths rule). The rule is "no proxy of the protected attribute as a feature".
- **Misuse**: 1,000 data requests, 20% for a purpose that the user did not consent to. The rule denies a request whose purpose is not in the user's consent. The check counts the uses without consent.

In [ ]:
# 13. Part J: retention, sovereignty, bias and misuse rules, each with a test that fails when the rule is off
NOW, DAY = 1_000_000_000, 86400
def part_j():
    rows = []
    def build(n=6000):
        r = np.random.default_rng(1); con = sqlite3.connect(":memory:"); data = [(i, f"u{i % 300}", int(NOW - t)) for i, t in enumerate(r.integers(0, 400 * DAY, n))]
        for t in ("events", "features"): con.execute(f"CREATE TABLE {t}(id INTEGER, user TEXT, ts INTEGER)"); con.executemany(f"INSERT INTO {t} VALUES (?,?,?)", data)
        return con
    def purge(con, days, tables):
        cut = NOW - days * DAY; ids = sorted(r[0] for r in con.execute("SELECT id FROM events WHERE ts<?", (cut,)))
        for t in tables: con.execute(f"DELETE FROM {t} WHERE ts<?", (cut,))
        return dict(cutoff=cut, rows=len(ids), sha256=hashlib.sha256(",".join(map(str, ids)).encode()).hexdigest()[:16])
    old = lambda con, days=90: sum(con.execute(f"SELECT COUNT(*) FROM {t} WHERE ts<?", (NOW - days * DAY,)).fetchone()[0] for t in ("events", "features"))   # rows past retention in all tables
    c_on, c_none, c_half = build(), build(), build(); cert = purge(c_on, 90, ("events", "features")); purge(c_half, 90, ("events",))
    print("deletion certificate:", cert); rows += [dict(rule="retention 90 days", case="rule on", violations=old(c_on)), dict(rule="retention 90 days", case="off: no purge", violations=old(c_none)),
                                                  dict(rule="retention 90 days", case="off: derived table forgotten", violations=old(c_half))]
    def sovereignty(enforce):
        r = np.random.default_rng(2); recs = []
        for i in range(3000):
            home = "EU" if r.random() < .4 else "US"; u = r.random(); recs.append(dict(home=home, flag=None if u < .05 else home.lower() + " " if u < .10 else home))
        sent = [x for x in recs if not enforce or (x["flag"] or "").strip().upper() == "US"]
        return sum(x["home"] == "EU" for x in sent), sum(x["home"] == "US" for x in recs) - sum(x["home"] == "US" for x in sent)
    (v1, b1), (v0, b0) = sovereignty(True), sovereignty(False)
    rows += [dict(rule="sovereignty (EU stays in EU)", case="rule on", violations=v1, cost=f"{b1} US records blocked"), dict(rule="sovereignty (EU stays in EU)", case="off", violations=v0, cost="0")]
    def bias(drop_proxy):
        r = np.random.default_rng(3); n = 8000; g = r.integers(0, 2, n); zipc = g + r.normal(0, .5, n); inc = r.normal(0, 1, n); hist = (inc - 0.8 * g + r.normal(0, .5, n)) > 0
        X = inc[:, None] if drop_proxy else np.c_[inc, zipc]; sel = LogisticRegression().fit(X[:4000], hist[:4000]).predict(X[4000:]); gg = g[4000:]
        return sel[gg == 1].mean() / sel[gg == 0].mean(), accuracy_score(hist[4000:], sel)
    (rt1, ac1), (rt0, ac0) = bias(True), bias(False)
    rows += [dict(rule="bias (no proxy feature)", case="rule on", violations=int(rt1 < 0.8), cost=f"ratio {rt1:.2f}, accuracy {ac1:.3f}"), dict(rule="bias (no proxy feature)", case="off", violations=int(rt0 < 0.8), cost=f"ratio {rt0:.2f}, accuracy {ac0:.3f}")]
    def misuse(enforce):
        r = np.random.default_rng(4); consent = {u: {"service", "analytics"} if u % 3 else {"service"} for u in range(300)}
        reqs = [(int(r.integers(0, 300)), str(r.choice(["service", "analytics", "ad_targeting"], p=[.6, .2, .2]))) for _ in range(1000)]
        ok = [(u, p) for u, p in reqs if not enforce or p in consent[u]]; return sum(p not in consent[u] for u, p in ok), len(reqs) - len(ok)
    (m1, d1), (m0, d0) = misuse(True), misuse(False)
    rows += [dict(rule="misuse (purpose limit)", case="rule on", violations=m1, cost=f"{d1} requests denied"), dict(rule="misuse (purpose limit)", case="off", violations=m0, cost="0")]
    t = pd.DataFrame(rows).fillna(""); t["check_passes"] = t.violations == 0; show("J_rules", t)
    for rule, g in t.groupby("rule", sort=False):
        on, off = g[g.case == "rule on"].check_passes.iloc[0], g[g.case.str.startswith("off")].check_passes
        CHECKS.append(dict(check=f"rule test: {rule}", passes_on_good=bool(on), passes_on_bad=bool(off.any()))); assert on and not off.any(), f"the test for '{rule}' cannot fail"
    RAW.append(dict(part="J", rows=rows, certificate=cert))
part("J rules on data", part_j)

In [ ]:
# 14. Status of every part, self-checks, raw rows. Stop the Kafka broker and any install. Print GPU memory.
b = S.get("broker")
if b is not None and b.poll() is None:
    try: os.killpg(b.pid, signal.SIGTERM); b.wait(timeout=60)
    except Exception: os.killpg(b.pid, signal.SIGKILL)
    print("Kafka broker stopped")
for name, p in PROCS.items():
    if p.poll() is None: os.killpg(p.pid, signal.SIGKILL); print("stopped background job", name)
show("status", pd.DataFrame(STATUS)); show("self_checks", pd.DataFrame(CHECKS))
(OUT / f"{SLUG}_raw_rows.json").write_text(json.dumps(RAW, default=str))
try: print("GPU memory in use (MiB):", subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30).stdout.split() or "no GPU found")
except Exception: print("GPU memory in use: no nvidia-smi (this notebook does not use the GPU)")
print(f"total wall time {(time.time() - T0) / 60:.1f} min | failed parts: {[s['part'] for s in STATUS if s['status'] != 'ok'] or 'none'}")

## What to look for

- **A.** Compare `exact` and `field_acc` per shape and per parser version. The `unparsed`, `partial` and `wrong` counts are separate outcomes. The `lines` column counts the lines of that one function. A second version often calls the first. The query table shows what you can ask of each shape before you write any parser.
- **B.** Read `dup_rows` and `ids_lost` together. A killed job with no key repeats rows. A job with a key does not. Both designs lose no ID. In the ETL against ELT table, read `rule_change_s`: ELT re-runs SQL on the raw rows it kept. The data here is small, so the times show the shape, not a benchmark.
- **C.** Compare `fix_only_this` with `fix_all_but_this`. A fault can matter little alone and a lot when the others are fixed, or the other way round. The leak row shows the training score against the clean production score.
- **D1.** Compare `cv_all_rows` with the next month, and `cv_inside_fold` with the next month. The gap, not the score, shows the leak. **D2.** Compare the flag encodings on the tree and on the linear model. A tree can rebuild a pattern from eight flags. A linear model cannot. The integer mask gives it a group key for a target mean. Dropping rows trains on a small, unusual sample.
- **E.** Naive-join cross-validation scores can look good, and the same model scores worse on the features that serving gives. The `rows_with_future_feature` column counts the leak. The skew table compares one pipeline computed two ways. The Feast rows appear only when the Feast venv installed.
- **F.** Dawid-Skene estimates an annotator's error pattern, so it can use an annotator who is wrong most of the time. Majority vote cannot. With 8 seeds and 200 labels, active learning shows a small gap with a wide interval. Read the interval, not only the mean.
- **G.** Read `mcnemar_p_vs_previous` for each step. The bigram "not good" needs "not" in the text. Removing stop words changes the token count and the vocabulary together.
- **H.** H1: the `fake` rows and the `kafka` rows must agree. If no `kafka` row exists, read the line "real Kafka broker" above the tables. H2: the allowed lateness trades freshness for exact windows. The last 3 hours are the measurement, the first 3 hours chose the setting. H3: on one machine, pandas and Spark local mode are close. Spark pays a startup cost and wins on scale-out, which this run does not test.
- **I.** The `meets` column compares a measured p95 with an assumed need. Change the assumption and the answer changes.
- **J.** Every rule has `violations` = 0 when on and more than 0 when off. The `cost` column is what the rule takes away (blocked US records, denied requests, lower accuracy).
- **Limits.** The data is simulated, so the effects are real in the simulation, not a forecast for your data. Single runs except D1 (5 seeds) and F active learning (8 seeds). The Spark and Kafka brokers run on one machine. Databricks and RAPIDS are not run here.

## Exam rules this proves

1. **data.6**: A SQL table is structured. Nested JSON is semi-structured: it has a tree shape, and a renamed key breaks a strict parser. A free-text email is unstructured: it needs parsing rules, and each rule has a hit rate.
2. **data.3**: Ingestion validates each record and sends the failures to a dead-letter queue, so no record is lost. A job that restarts repeats records unless the write uses an idempotency key. ETL transforms before the load. ELT loads raw data first and keeps it for later rules.
3. **data.1 (EDA)**: Look for unit mixes, skew, missing values that depend on the target, imbalance and leaking columns with code, before you model.
4. **data.1 (feature engineering)**: Compute a feature that uses the target (a target mean) inside each training fold. Keep the missing pattern as a feature. One integer per pattern (flag times 2 to the power of the column index, summed) also works as a group key.
5. **data.1 (feature store)**: A point-in-time join takes the value that was known at each label's time. A naive join leaks the future. One shared function for training and serving removes training-serving skew. An online store reads one row by key in well under a millisecond.
6. **data.4**: Several annotators with known error rates give better labels with Dawid-Skene than with a vote. Weak rules and active learning save labels. A text step can hurt: a stop-word list that removes "not" changes the meaning.
7. **data.10**: Training reads many rows in batch. Inference reads one row by key in milliseconds. Retrieval searches an index that a batch job built. A stream must choose between freshness and exact windows when events arrive late.
8. **data.3 (Kafka, Spark)**: A key goes to one partition. A consumer group shares the partitions. At-least-once delivery repeats records after a crash, and an idempotent sink hides that. At-most-once delivery loses records. Spark local mode shows engine overhead. Its value is scale-out.
9. **data.5**: A rule on data needs a check of the final state that fails when the rule is off: retention in every table, no EU record in the global store, no proxy of a protected attribute, no use beyond consent.

| Objective | Exam term | One plain line | Shown in |
|---|---|---|---|
| data.1 | Feature engineering | Build model inputs from raw data: logs, ratios, the missing-pattern mask, target means inside the fold. | D1, D2, D3 |
| data.1 | Feature stores | One definition of each feature, an offline copy for training and an online copy for serving (source: `what-is-a-feature-store.txt`). | E |
| data.1 | Exploratory data analysis | Check the data with code before you model: five planted faults. | C |
| data.3 | Data ingestion | Read records from a source into your system, with validation and a dead-letter queue. | B, H1 |
| data.3 | ETL components | Extract, transform, load: validation, a dead-letter queue, the load with a key. | B |
| data.3 | Distributed processing | Spread one job over many workers. Spark local mode uses 4 cores of one machine. | H3 |
| data.3 | Spark | An engine for data jobs on many cores or machines. Run here in local mode. | H3 |
| data.3 | Databricks | A data platform that runs Spark. The NVIDIA blog compares Databricks Photon on CPUs with the RAPIDS Accelerator on GPUs (source: `gpus-for-etl-run-faster-less-costly-workloads-with-nvidia-rapids-accelerator-for-apache-spark-and-databricks.txt`). Not run: needs an account. | text only |
| data.3 | Kafka | A log of events in partitions. Producers write, consumers read (source: `intro.txt`). Run as a real KRaft broker, with an in-memory fallback. | H1 |
| data.3 | NVIDIA RAPIDS | The same blog says the RAPIDS Accelerator for Apache Spark runs Spark SQL on GPUs with no change to the Spark SQL code. Not run: this notebook is CPU only. `data-flywheel-cudf.ipynb` measures cuDF. | text only |
| data.4 | Data preprocessing | Shape raw data for a model: parse, validate, normalise, fix units. | A, B, C, G |
| data.4 | Data cleaning | Remove or fix faults: duplicates, bad units, bad records. | B, C, G |
| data.4 | Data labeling | Get labels from annotators, rules or active learning. Labels from people have errors (source: `integrating-with-data-generation-and-labeling-tools-for-accurate-ai-training.txt`). | F |
| data.4 | Text preprocessing | Normalise, deduplicate, tokenize, remove stop words. | G |
| data.5 | Privacy requirements | Rules that limit how long and where data lives, and what it is used for. | J |
| data.5 | PII handling | Find personal data and redact it. Covered in `data-flywheel-cudf.ipynb`, not repeated. | text only |
| data.5 | Data retention | Delete data after a set time, in every table, with a certificate. | J |
| data.5 | Data sovereignty | Data stays in the region whose law applies. A bad flag is treated as EU. | J |
| data.5 | Bias | A model can treat groups unequally through a proxy. The four-fifths ratio measures it. | J |
| data.5 | Misuse risks | Data used beyond the purpose a person agreed to. A purpose check denies it. | J |
| data.6 | Structured | Rows and columns with a fixed schema. | A |
| data.6 | Semi-structured | Tree-shaped records with no fixed schema. | A |
| data.6 | Unstructured | Free text with no schema. | A |
| data.10 | Training data flow | Many rows, read in batch, hours of time budget. | I |
| data.10 | Inference data flow | One row by key, milliseconds, for each request. | I, E |
| data.10 | Retrieval data flow | A question searches an index that a batch job built. | I |
| data.10 | Batch versus streaming | Batch recompute is exact but late. Streaming is fresh but drops late events unless you wait. | H2, I |